# V1 No Geopraphies

In [ ]:
# ── Colab: 3D interactive network from CSV → standalone HTML ──
import pandas as pd, re, json, difflib, io
from collections import Counter, defaultdict

# ───────── CONFIG ─────────
CSV_FILE   = None        # None = upload dialog; or e.g. "data.csv" / raw GitHub URL
OUT_HTML   = "network_3d.html"
HUB_COL    = "PI"        # default hub column for "hub" link mode
MERGE_ACROSS_COLS = False  # True: same value in different columns (e.g. State=Georgia, AOI=Georgia) = ONE node
FUZZY      = 0.88        # near-duplicate merge threshold within a column (1.0 = off). Skips values with digits.
ALIASES    = {           # manual merges: variant -> canonical (case/punct-insensitive)
    # "GOES-R": "GOES-ABI",
    # "Gerogia": "Georgia",
}
NA = {"", "nan", "none", "n/a", "na", "-", "tbd", "null"}
PALETTE = ["#e63946","#f4a261","#2a9d8f","#4361ee","#e9c46a","#9b5de5","#00bbf9","#f15bb5",
           "#80ed99","#ff9f1c","#06d6a0","#c77dff","#ef476f","#118ab2","#ffd166","#8ac926",
           "#ff595e","#6a994e","#bc6c25","#48cae4"]

# ───────── LOAD ─────────
if CSV_FILE is None:
    from google.colab import files
    up = files.upload(); CSV_FILE = next(iter(up)); raw_bytes = up[CSV_FILE]
else:
    raw_bytes = open(CSV_FILE, "rb").read() if not CSV_FILE.startswith("http") else None
try:    text = raw_bytes.decode("utf-8-sig") if raw_bytes else None
except UnicodeDecodeError: text = raw_bytes.decode("latin-1")
head = (text or "").split("\n", 1)[0]
sep = "\t" if "\t" in head else ","
df = pd.read_csv(io.StringIO(text) if text else CSV_FILE, sep=sep, dtype=str, keep_default_na=False)
df.columns = [c.strip() for c in df.columns]
COLS = list(df.columns)
HUB_COL = HUB_COL if HUB_COL in COLS else COLS[0]
print(f"{len(df)} rows | sep={'TAB' if sep=='\t' else 'COMMA'} | cols={COLS}")

# ───────── PARSE / NORMALIZE ─────────
STOP = {"of","and","the","for","in","on","a","an","&"}
norm = lambda s: re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()
ALIAS_N = {norm(k): v for k, v in ALIASES.items()}

def split_cell(v):
    v = str(v).strip()
    out = []
    for p in re.split(r"[;\n|]|,(?![^()]*\))", v):      # commas inside (...) are kept
        p = re.sub(r"^\s*(and|&)\s+", "", p.strip(), flags=re.I).strip(" ,.;:")
        while p.endswith(")") and p.count(")") > p.count("("): p = p[:-1].strip()
        while p.startswith("(") and p.count("(") > p.count(")"): p = p[1:].strip()
        if p.lower() in NA: continue
        out.append(ALIAS_N.get(norm(p), p))
    return out

def parse(p):
    """'High Resolution Rapid Refresh (HRRR)' -> (base, acronym)"""
    m = re.match(r"^(.*?)\s*\(([^()]+)\)\s*$", p)
    if m and m.group(1) and re.fullmatch(r"[A-Za-z0-9&\-]{2,10}", m.group(2)) \
       and sum(ch.isupper() for ch in m.group(2)) >= 2:
        return m.group(1).strip(), m.group(2)
    return p, None

class UF:
    def __init__(s): s.p = {}
    def f(s, x):
        s.p.setdefault(x, x)
        while s.p[x] != x: s.p[x] = s.p[s.p[x]]; x = s.p[x]
        return x
    def u(s, a, b): s.p[s.f(a)] = s.f(b)

parsed = [{c: [(p, *parse(p)) for p in split_cell(r[c])] for c in COLS} for _, r in df.iterrows()]

ufs = {}
for c in COLS:
    uf = UF(); keys = set()
    for row in parsed:
        for raw, base, acr in row[c]:
            kb = norm(base); keys.add(kb); uf.f(kb)
            if acr: ka = norm(acr); keys.add(ka); uf.u(kb, ka)
    for k in list(keys):                                  # "Federal Emergency Mgmt Agency" ↔ "FEMA" via initials
        w = k.split()
        if len(w) >= 2:
            for ini in {"".join(x[0] for x in w if x not in STOP), "".join(x[0] for x in w)}:
                if len(ini) >= 2 and ini in keys and ini != k: uf.u(k, ini)
    if FUZZY < 1:
        ks = sorted(k for k in keys if len(k) >= 5 and not re.search(r"\d", k))
        for i, a in enumerate(ks):
            for b in ks[i+1:]:
                if abs(len(a)-len(b)) > 3: continue
                r = difflib.SequenceMatcher(None, a, b).ratio()
                if r >= FUZZY and uf.f(a) != uf.f(b):
                    uf.u(a, b); print(f"  fuzzy merge [{c}]: '{a}' ~ '{b}' ({r:.2f})")
    ufs[c] = uf

# ───────── BUILD NODES / ROWS ─────────
nodes, variants, rows = {}, defaultdict(Counter), []
for row in parsed:
    ids = []
    for c in COLS:
        for raw, base, acr in row[c]:
            key = ufs[c].f(norm(base))
            nid = key if MERGE_ACROSS_COLS else f"{c}::{key}"
            nodes.setdefault(nid, {"id": nid, "col": c, "rows": 0})
            variants[nid][raw] += 1
            if nid not in ids: ids.append(nid)
    for nid in ids: nodes[nid]["rows"] += 1
    rows.append(ids)
for nid, n in nodes.items():
    vs = variants[nid]
    n["label"] = max(vs, key=lambda s: (len(s), vs[s]))
    n["aliases"] = sorted(v for v in vs if v != n["label"])
colors = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(COLS)}
counts = Counter(n["col"] for n in nodes.values())
data = {"cols": COLS, "colors": colors, "counts": {c: counts[c] for c in COLS},
        "hub": HUB_COL, "nodes": list(nodes.values()), "rows": rows}

# ───────── HTML (3d-force-graph / three.js) ─────────
HTML = r"""<!doctype html><html><head><meta charset="utf-8"><title>Network 3D</title>
<style>
html,body{margin:0;height:100%;background:#0b0e14;color:#dde3ee;font:12px/1.35 system-ui,sans-serif;overflow:hidden}
#graph{position:fixed;inset:0}
.p{position:fixed;top:8px;background:#141923ee;border:1px solid #2a3142;border-radius:8px;padding:10px;max-height:calc(100vh - 36px);overflow-y:auto}
#panel{left:8px;width:262px}#info{right:8px;width:290px;display:none}
h4{margin:10px 0 4px;font-size:11px;text-transform:uppercase;letter-spacing:.06em;color:#8a93a6}
h4:first-child{margin-top:0}
label.col{display:flex;align-items:center;gap:6px;padding:1px 0;cursor:pointer}
label.col i{width:10px;height:10px;border-radius:50%;display:inline-block}
label.col span{margin-left:auto;color:#6b7488}
input[type=range]{width:100%}select,input[type=text],button{background:#1d2431;color:#dde3ee;border:1px solid #333c50;border-radius:5px;padding:4px 6px;font-size:12px}
select,input[type=text]{width:100%;box-sizing:border-box}button{cursor:pointer}button:hover{background:#2a3446}
.row{display:flex;gap:4px;margin:3px 0}.row>*{flex:1}
.v{float:right;color:#8a93a6}
#info a{color:#7cc4ff;cursor:pointer;display:block;padding:1px 0}#info a:hover{text-decoration:underline}
#stats{position:fixed;bottom:8px;left:50%;transform:translateX(-50%);background:#141923cc;padding:4px 10px;border-radius:12px;color:#8a93a6}
</style></head><body>
<div id="graph"></div>
<div id="panel" class="p">
 <h4>Search</h4><input type="text" id="q" list="dl" placeholder="node / alias, Enter"><datalist id="dl"></datalist>
 <h4>Columns (toggle nodes) <span class="v"><a href="#" id="all">all</a> · <a href="#" id="none">none</a></span></h4><div id="legend"></div>
 <h4>Link mode</h4>
 <select id="mode"><option value="all">All pairs in a row (clique)</option><option value="hub">Hub column → everything</option><option value="chain">Chain by column order</option></select>
 <div class="row"><select id="hub"></select></div>
 <label class="col"><input type="checkbox" id="intra" checked> link values within same column</label>
 <h4>Filters</h4>
 <div>Min edge weight (co-occurrence) <span class="v" id="vW">1</span></div><input type="range" id="minW" min="1" max="20" value="1">
 <div>Min node degree <span class="v" id="vD">0</span></div><input type="range" id="minDeg" min="0" max="30" value="0">
 <h4>Appearance</h4>
 <label class="col"><input type="checkbox" id="labels"> text labels</label>
 <div>Label min degree <span class="v" id="vL">0</span></div><input type="range" id="labelDeg" min="0" max="30" value="0">
 <div>Node size</div><input type="range" id="size" min="0.3" max="4" step="0.1" value="1">
 <div>Link opacity</div><input type="range" id="op" min="0.05" max="1" step="0.05" value="0.35">
 <label class="col"><input type="checkbox" id="part"> flow particles</label>
 <label class="col"><input type="checkbox" id="rot"> auto-rotate</label>
 <h4>Physics</h4>
 <div>Link distance</div><input type="range" id="dist" min="10" max="300" value="40">
 <div>Repulsion</div><input type="range" id="chg" min="5" max="400" value="30">
 <div class="row"><button id="dim">2D / 3D</button><button id="fit">Fit view</button><button id="snap">PNG</button></div>
 <div style="color:#6b7488;margin-top:6px">left-drag rotate · right-drag pan · wheel zoom · click node = focus + highlight</div>
</div>
<div id="info" class="p"></div><div id="stats"></div>
<script src="https://unpkg.com/three@0.150.1/build/three.min.js"></script>
<script src="https://unpkg.com/three-spritetext@1.8.0/dist/three-spritetext.min.js"></script>
<script src="https://unpkg.com/3d-force-graph@1.73.2/dist/3d-force-graph.min.js"></script>
<script>
const D=__DATA__, $=id=>document.getElementById(id);
const esc=s=>String(s).replace(/[&<>"]/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));
const NODE=new Map(D.nodes.map(n=>[n.id,n])), cache=new Map(), CUR={links:[],nodes:[]};
const S={vis:new Set(D.cols),mode:'all',hub:D.hub,minW:1,minDeg:0,intra:true,labels:false,labelDeg:0,size:1,hl:null,sel:null,dim:3};
const sid=x=>typeof x==='object'?x.id:x;
function labelObj(n){
  if(!S.labels||n.deg<S.labelDeg||typeof SpriteText==='undefined')return null;
  const s=new SpriteText(n.label,3.2,D.colors[n.col]);
  s.position.y=Math.cbrt(Math.max(1,Math.sqrt(n.deg+1))*S.size)*4+4; s.material.depthWrite=false; return s;
}
const G=ForceGraph3D({controlType:'trackball'})($('graph')).backgroundColor('#0b0e14').nodeRelSize(4)
 .nodeVal(n=>Math.max(1,Math.sqrt(n.deg+1))*S.size)
 .nodeColor(n=>S.hl&&!S.hl.n.has(n.id)?'#262b36':D.colors[n.col])
 .nodeLabel(n=>`<div style="background:#000d;padding:4px 8px;border-radius:4px;font:12px system-ui"><b>${esc(n.label)}</b><br>${esc(n.col)} · ${n.deg} links · ${n.rows} rows${n.aliases.length?'<br><i>aka '+esc(n.aliases.join(', '))+'</i>':''}</div>`)
 .linkColor(l=>S.hl&&!S.hl.l.has(l)?'#1b1f29':'#8a93a6').linkOpacity(.35)
 .linkWidth(l=>S.hl&&S.hl.l.has(l)?2:Math.min(.3+l.w*.35,4))
 .linkDirectionalParticleWidth(1.6)
 .nodeThreeObjectExtend(true).nodeThreeObject(labelObj)
 .onNodeClick(pick).onBackgroundClick(()=>select(null));
let fitted=false; G.onEngineStop(()=>{if(!fitted){fitted=true;G.zoomToFit(600,60)}});
const refresh=()=>{G.nodeColor(G.nodeColor());G.linkColor(G.linkColor());G.linkWidth(G.linkWidth())};

function build(){
  const W=new Map(), add=(a,b)=>{if(a===b)return;const k=a<b?a+'\u0001'+b:b+'\u0001'+a;W.set(k,(W.get(k)||0)+1)};
  const clique=L=>{for(let i=0;i<L.length;i++)for(let j=i+1;j<L.length;j++)add(L[i],L[j])};
  for(const row of D.rows){
    const ids=row.filter(i=>S.vis.has(NODE.get(i).col)), col=i=>NODE.get(i).col;
    if(S.mode==='all'){
      for(let i=0;i<ids.length;i++)for(let j=i+1;j<ids.length;j++){
        if(!S.intra&&col(ids[i])===col(ids[j]))continue; add(ids[i],ids[j]);}
    }else if(S.mode==='hub'){
      const h=ids.filter(i=>col(i)===S.hub), r=ids.filter(i=>col(i)!==S.hub);
      h.forEach(a=>r.forEach(b=>add(a,b)));
    }else{
      const by={}; ids.forEach(i=>(by[col(i)]??=[]).push(i));
      const cs=D.cols.filter(c=>by[c]);
      for(let k=0;k<cs.length-1;k++)by[cs[k]].forEach(a=>by[cs[k+1]].forEach(b=>add(a,b)));
      if(S.intra)cs.forEach(c=>clique(by[c]));
    }
  }
  const raw=[],deg=new Map();
  for(const [k,w] of W){ if(w<S.minW)continue; const [a,b]=k.split('\u0001'); raw.push({source:a,target:b,w});
    deg.set(a,(deg.get(a)||0)+1); deg.set(b,(deg.get(b)||0)+1); }
  const nodes=D.nodes.filter(n=>S.vis.has(n.col)&&(deg.get(n.id)||0)>=S.minDeg).map(n=>{
    let o=cache.get(n.id); if(!o){o={...n};cache.set(n.id,o)} o.deg=deg.get(n.id)||0; return o;});
  const keep=new Set(nodes.map(n=>n.id));
  const links=raw.filter(l=>keep.has(l.source)&&keep.has(l.target));
  CUR.links=links; CUR.nodes=nodes;
  G.graphData({nodes,links}); G.nodeThreeObject(labelObj);
  $('stats').textContent=`${nodes.length} nodes · ${links.length} edges`;
  const s=S.sel&&cache.get(S.sel); if(s&&keep.has(s.id))select(s); else select(null);
}
function select(n){
  S.sel=n?n.id:null;
  if(!n){S.hl=null;$('info').style.display='none';refresh();return}
  const ns=new Set([n.id]),ls=new Set(),by={};
  for(const l of CUR.links){const a=sid(l.source),b=sid(l.target);
    if(a===n.id||b===n.id){ls.add(l);const o=a===n.id?b:a;ns.add(o);(by[NODE.get(o).col]??=[]).push([o,l.w]);}}
  S.hl={n:ns,l:ls};
  let h=`<h4>${esc(n.col)}</h4><b style="font-size:14px;color:${D.colors[n.col]}">${esc(n.label)}</b>`;
  if(n.aliases.length)h+=`<div><i>aka ${esc(n.aliases.join(', '))}</i></div>`;
  h+=`<div>${n.rows} rows · ${ns.size-1} connections</div>`;
  for(const c of D.cols){const L=by[c]; if(!L)continue; L.sort((a,b)=>b[1]-a[1]);
    h+=`<h4><span style="color:${D.colors[c]}">●</span> ${esc(c)} (${L.length})</h4>`+L.map(([o,w])=>`<a data-id="${esc(o)}">${esc(NODE.get(o).label)} <span style="color:#6b7488">×${w}</span></a>`).join('');}
  $('info').innerHTML=h; $('info').style.display='block'; refresh();
}
function pick(n){ select(n); if(!n)return; const x=n.x||0,y=n.y||0,z=n.z||0,d=Math.hypot(x,y,z)||1,r=1+120/d;
  G.cameraPosition({x:x*r,y:y*r,z:z*r},{x,y,z},800); }
$('info').onclick=e=>{const o=cache.get(e.target.dataset.id); if(o)pick(o)};

// ── UI wiring ──
D.cols.forEach(c=>{const l=document.createElement('label');l.className='col';
  l.innerHTML=`<input type="checkbox" checked data-c="${esc(c)}"><i style="background:${D.colors[c]}"></i>${esc(c)}<span>${D.counts[c]}</span>`;$('legend').append(l);
  const o=document.createElement('option');o.value=o.textContent=c;if(c===D.hub)o.selected=true;$('hub').append(o);});
D.nodes.forEach(n=>{const o=document.createElement('option');o.value=n.label;$('dl').append(o)});
$('legend').onchange=e=>{const c=e.target.dataset.c;e.target.checked?S.vis.add(c):S.vis.delete(c);build()};
const setAll=v=>{document.querySelectorAll('#legend input').forEach(i=>{i.checked=v;v?S.vis.add(i.dataset.c):S.vis.delete(i.dataset.c)});build()};
$('all').onclick=e=>{e.preventDefault();setAll(true)}; $('none').onclick=e=>{e.preventDefault();setAll(false)};
$('mode').onchange=e=>{S.mode=e.target.value;build()}; $('hub').onchange=e=>{S.hub=e.target.value;build()};
$('intra').onchange=e=>{S.intra=e.target.checked;build()};
$('minW').oninput=e=>{S.minW=+e.target.value;$('vW').textContent=S.minW;build()};
$('minDeg').oninput=e=>{S.minDeg=+e.target.value;$('vD').textContent=S.minDeg;build()};
$('labels').onchange=e=>{S.labels=e.target.checked;G.nodeThreeObject(labelObj)};
$('labelDeg').oninput=e=>{S.labelDeg=+e.target.value;$('vL').textContent=S.labelDeg;G.nodeThreeObject(labelObj)};
$('size').oninput=e=>{S.size=+e.target.value;G.nodeVal(G.nodeVal());G.nodeThreeObject(labelObj)};
$('op').oninput=e=>G.linkOpacity(+e.target.value);
$('part').onchange=e=>G.linkDirectionalParticles(e.target.checked?2:0);
$('dist').oninput=e=>{G.d3Force('link').distance(+e.target.value);G.d3ReheatSimulation()};
$('chg').oninput=e=>{G.d3Force('charge').strength(-+e.target.value);G.d3ReheatSimulation()};
let rot=null; $('rot').onchange=e=>{clearInterval(rot); if(e.target.checked)rot=setInterval(()=>{
  const p=G.cameraPosition(),c=Math.cos(.004),s=Math.sin(.004); G.cameraPosition({x:p.x*c+p.z*s,y:p.y,z:-p.x*s+p.z*c});},30)};
$('dim').onclick=()=>{S.dim=S.dim===3?2:3;G.numDimensions(S.dim);G.d3ReheatSimulation()};
$('fit').onclick=()=>G.zoomToFit(600,60);
$('snap').onclick=()=>{const r=G.renderer();r.render(G.scene(),G.camera());const a=document.createElement('a');a.href=r.domElement.toDataURL('image/png');a.download='network.png';a.click()};
$('q').addEventListener('change',e=>{
  const q=e.target.value.trim().toLowerCase(); if(!q)return;
  const n=D.nodes.find(x=>x.label.toLowerCase()===q)||D.nodes.find(x=>(x.label+' '+x.aliases.join(' ')).toLowerCase().includes(q));
  if(!n){$('stats').textContent='no match';return}
  if(!S.vis.has(n.col)){S.vis.add(n.col);document.querySelector(`#legend input[data-c="${CSS.escape(n.col)}"]`).checked=true;build();}
  setTimeout(()=>{const o=cache.get(n.id); CUR.nodes.includes(o)?pick(o):$('stats').textContent='node hidden by filters'},300);});
G.d3Force('link').distance(40); G.d3Force('charge').strength(-30);
build();
</script></body></html>"""

html = HTML.replace("__DATA__", json.dumps(data, ensure_ascii=False).replace("</", "<\\/"))
open(OUT_HTML, "w", encoding="utf-8").write(html)
print(f"✔ {OUT_HTML}: {len(nodes)} nodes after merging")
try:
    from google.colab import files; files.download(OUT_HTML)
except Exception: pass

Saving Network_Viz_A9.csv to Network_Viz_A9 (1).csv
999 rows | sep=COMMA | cols=['PI', 'EA Element', 'Institution', 'State', 'Model', 'Framework', 'Task', 'Output', 'Sensor and Instruments', 'Area of Interest', 'Area of Interest_2', 'Temporal', 'Temporal_2', 'Prediction scale', 'End users', 'End user sector', 'Synergies']
  fuzzy merge [Area of Interest_2]: 'california' ~ 'califronia' (0.90)
  fuzzy merge [End users]: 'us forest service' ~ 'usda forest service' (0.94)
✔ network_3d.html: 279 nodes after merging


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# V2 with Geographies

In [4]:
# ── Network Viz v2.3: network + geography + End users placed by YOUR "<col> Location" field ──
import pandas as pd, re, json, difflib, io, os, time, unicodedata, urllib.request, urllib.parse
from collections import Counter, defaultdict
from datetime import date, timedelta

# ───────── CONFIG ─────────
CSV_FILE   = None                       # None = upload dialog; or path / raw GitHub URL
OUT_HTML   = "network_v2.html"
HUB_COL    = "PI"
GEO_COLS   = ["State", "Area of Interest", "Area of Interest_2"]   # polygon geographies (states / regions / countries)
ORG_COLS   = ["End users"]              # org columns. Each uses "<col> Location" if that column exists (e.g. "End users Location")
LOC_SUFFIX = " Location"
MERGE_ACROSS_COLS = False
FUZZY      = 0.88                       # near-dup merge inside a column (1.0 = off)
GEO_FUZZY  = 0.80
ALIASES    = {}                         # {"GOES-R": "GOES-ABI", ...}
GEOCODE_ONLINE = True                   # turn the location TEXT into lat/lon via OpenStreetMap Nominatim (cached, 1 req/sec)
GEOCODE_CACHE  = "geocode_cache.json"
LOCATIONS  = {}                         # hard overrides: {"Phillips 66": (29.744, -95.598, "Houston, TX")}  beats the CSV
IMAGERY_DATE = (date.today() - timedelta(days=2)).isoformat()

GIBS_BASE = [
    ("NASA Blue Marble: shaded relief + bathymetry", "BlueMarble_ShadedRelief_Bathymetry", "GoogleMapsCompatible_Level8", "jpeg", None),
    ("NASA Blue Marble: Next Generation",            "BlueMarble_NextGeneration",          "GoogleMapsCompatible_Level8", "jpeg", None),
    ("NASA MODIS Terra true color (daily)",          "MODIS_Terra_CorrectedReflectance_TrueColor", "GoogleMapsCompatible_Level9", "jpeg", "date"),
    ("NASA VIIRS SNPP true color (daily)",           "VIIRS_SNPP_CorrectedReflectance_TrueColor",  "GoogleMapsCompatible_Level9", "jpeg", "date"),
]
GIBS_MERGED = [
    ("NASA merged: Blue Marble + VIIRS true color (gap-filled)", ["BlueMarble_NextGeneration", "VIIRS_SNPP_CorrectedReflectance_TrueColor"]),
    ("NASA merged: Blue Marble + MODIS Terra (gap-filled)",      ["BlueMarble_NextGeneration", "MODIS_Terra_CorrectedReflectance_TrueColor"]),
]
GIBS_OVER = [
    ("NASA borders & features (reference)", "Reference_Features_15m", "GoogleMapsCompatible_Level13", "png", None, False, False),
    ("NASA coastlines (reference)",         "Coastlines_15m",         "GoogleMapsCompatible_Level13", "png", None, False, False),
    ("NASA place labels (reference)",       "Reference_Labels_15m",   "GoogleMapsCompatible_Level13", "png", None, False, False),
    ("Land surface temp, day",              "MODIS_Terra_Land_Surface_Temp_Day", "GoogleMapsCompatible_Level7", "png", "date", True, False),
    ("Aerosol optical depth",               "MODIS_Combined_Value_Added_AOD",    "GoogleMapsCompatible_Level6", "png", "date", True, False),
    ("Snow cover",                          "MODIS_Terra_NDSI_Snow_Cover",       "GoogleMapsCompatible_Level8", "png", "date", True, False),
]
STATES_URLS = ["https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json",
               "https://raw.githubusercontent.com/python-visualization/folium/main/examples/data/us-states.json"]
WORLD_URLS  = ["https://raw.githubusercontent.com/johan/world.geo.json/master/countries.geo.json",
               "https://raw.githubusercontent.com/python-visualization/folium/main/examples/data/world-countries.json"]
NA = {"", "nan", "none", "n/a", "na", "-", "tbd", "null"}
PALETTE = ["#e63946","#f4a261","#2a9d8f","#4361ee","#e9c46a","#9b5de5","#00bbf9","#f15bb5","#80ed99","#ff9f1c",
           "#06d6a0","#c77dff","#ef476f","#118ab2","#ffd166","#8ac926","#ff595e","#6a994e","#bc6c25","#48cae4"]

# ───────── polygon gazetteer ─────────
ABBR = ("AL Alabama,AK Alaska,AZ Arizona,AR Arkansas,CA California,CO Colorado,CT Connecticut,DE Delaware,DC District of Columbia,"
        "FL Florida,GA Georgia,HI Hawaii,ID Idaho,IL Illinois,IN Indiana,IA Iowa,KS Kansas,KY Kentucky,LA Louisiana,ME Maine,"
        "MD Maryland,MA Massachusetts,MI Michigan,MN Minnesota,MS Mississippi,MO Missouri,MT Montana,NE Nebraska,NV Nevada,"
        "NH New Hampshire,NJ New Jersey,NM New Mexico,NY New York,NC North Carolina,ND North Dakota,OH Ohio,OK Oklahoma,OR Oregon,"
        "PA Pennsylvania,PR Puerto Rico,RI Rhode Island,SC South Carolina,SD South Dakota,TN Tennessee,TX Texas,UT Utah,VT Vermont,"
        "VA Virginia,WA Washington,WV West Virginia,WI Wisconsin,WY Wyoming")
ab2name = dict(x.split(" ", 1) for x in ABBR.split(","))
R = lambda s: [ab2name[a] for a in s.split()]
ALL48 = [n for a, n in ab2name.items() if a not in ("AK", "HI", "PR")]
EAST_COAST = R("ME NH MA RI CT NY NJ DE MD VA NC SC GA FL")
GULF = R("TX LA MS AL FL")
MIDWEST = R("IL IN IA KS MI MN MO NE ND OH SD WI")
REGIONS = {
    "conus": ALL48, "contiguous us": ALL48, "contiguous united states": ALL48, "continental us": ALL48,
    "continental united states": ALL48, "lower 48": ALL48,
    "united states": ALL48 + ["Alaska", "Hawaii"], "us": ALL48 + ["Alaska", "Hawaii"], "usa": ALL48 + ["Alaska", "Hawaii"],
    "conus coastal": R("ME NH MA RI CT NY NJ DE MD VA NC SC GA FL AL MS LA TX CA OR WA"),
    "us east": R("ME NH VT MA RI CT NY NJ PA DE MD DC VA WV NC SC GA FL OH MI IN KY TN AL MS"),
    "east coast": EAST_COAST, "us east coast": EAST_COAST, "eastern seaboard": EAST_COAST,
    "west coast": R("CA OR WA"), "us west": R("WA OR CA NV ID UT AZ MT WY CO NM"),
    "gulf coast": GULF, "gulf coasts": GULF, "gulf states": GULF,
    "midwest": MIDWEST, "midwest united states": MIDWEST, "us midwest": MIDWEST, "midwestern us": MIDWEST,
    "northeast": R("ME NH VT MA RI CT NY NJ PA"), "southeast": R("AL FL GA KY MS NC SC TN VA WV"),
    "southwest": R("AZ NM TX OK"), "pacific northwest": R("WA OR ID"), "great lakes": R("MN WI IL IN MI OH PA NY"),
    "great plains": R("ND SD NE KS OK"),
}
GLOBAL = {"global", "worldwide", "world"}
COUNTRY_ALIAS = {"tanzania": ["United Republic of Tanzania", "Tanzania"], "uk": ["United Kingdom"],
                 "great britain": ["United Kingdom"], "ivory coast": ["Ivory Coast", "Côte d'Ivoire"]}

# ───────── LOAD ─────────
if CSV_FILE is None:
    from google.colab import files
    up = files.upload(); CSV_FILE = next(iter(up)); raw_bytes = up[CSV_FILE]
elif CSV_FILE.startswith("http"):
    raw_bytes = urllib.request.urlopen(CSV_FILE, timeout=30).read()
else:
    raw_bytes = open(CSV_FILE, "rb").read()
try:    text = raw_bytes.decode("utf-8-sig")
except UnicodeDecodeError: text = raw_bytes.decode("latin-1")
sep = "\t" if "\t" in text.split("\n", 1)[0] else ","
df = pd.read_csv(io.StringIO(text), sep=sep, dtype=str, keep_default_na=False)
df.columns = [c.strip() for c in df.columns]
lc = {c.lower(): c for c in df.columns}
GEO_COLS = [lc[g.lower()] for g in GEO_COLS if g.lower() in lc]
ORG_COLS = [lc[g.lower()] for g in ORG_COLS if g.lower() in lc]
LOC_FIELD = {c: lc[(c + LOC_SUFFIX).lower()] for c in ORG_COLS if (c + LOC_SUFFIX).lower() in lc}
COLS = [c for c in df.columns if c not in set(LOC_FIELD.values())]      # location columns are data, not nodes
HUB_COL = lc.get(HUB_COL.lower(), COLS[0])
print(len(df), "rows | sep =", "TAB" if sep == "\t" else "COMMA", "| geo cols:", GEO_COLS, "| org cols:", ORG_COLS, "| location fields:", LOC_FIELD)
for c in ORG_COLS:
    if c not in LOC_FIELD: print("  !! no '%s%s' column found -> those orgs can't be placed" % (c, LOC_SUFFIX))

# ───────── PARSE / NORMALIZE ─────────
STOP = {"of", "and", "the", "for", "in", "on", "a", "an", "&"}
def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", " ", s).strip()
ALIAS_N = {norm(k): v for k, v in ALIASES.items()}
SPLIT = re.compile(r"[;\n|]|(?<=\))\s+and\s+|,(?![^()]*\))")

def split_cell(v, org=False):
    parts = []
    for p in SPLIT.split(str(v).strip()):
        p = p.strip()
        if not p: continue
        if parts and (re.fullmatch(r"\([^()]*\)[\s,.;:]*", p) or (org and re.fullmatch(r"[A-Z]{2}[.:]?", p) and p[:2] in ab2name)):
            parts[-1] += (" " if p.startswith("(") else ", ") + p.rstrip(" ,.;:"); continue
        parts.append(p)
    out = []
    for p in parts:
        p = re.sub(r"^\s*(and|&)\s+", "", p, flags=re.I).strip(" ,.;:")
        p = re.sub(r"[’']s$", "", p)
        while p.endswith(")") and p.count(")") > p.count("("): p = p[:-1].strip()
        while p.startswith("(") and p.count("(") > p.count(")"): p = p[1:].strip()
        if p.lower() in NA: continue
        out.append(ALIAS_N.get(norm(p), p))
    return out

def parse(p):
    m = re.match(r"^(.*?)\s*\(([^()]+)\)\s*$", p)
    if m and m.group(1) and re.fullmatch(r"[A-Za-z0-9&\-]{2,10}", m.group(2)) and sum(ch.isupper() for ch in m.group(2)) >= 2:
        return m.group(1).strip(), m.group(2)
    return p, None

class UF:
    def __init__(s): s.p = {}
    def f(s, x):
        s.p.setdefault(x, x)
        while s.p[x] != x: s.p[x] = s.p[s.p[x]]; x = s.p[x]
        return x
    def u(s, a, b): s.p[s.f(a)] = s.f(b)

parsed = [{c: [(p, *parse(p)) for p in split_cell(r[c], c in ORG_COLS)] for c in COLS} for _, r in df.iterrows()]
ufs = {}
for c in COLS:
    uf = UF(); keys = set()
    for row in parsed:
        for raw, base, acr in row[c]:
            kb = norm(base); keys.add(kb); uf.f(kb)
            if acr: ka = norm(acr); keys.add(ka); uf.u(kb, ka)
    for k in list(keys):
        w = k.split()
        if len(w) >= 2:
            for ini in {"".join(x[0] for x in w if x not in STOP), "".join(x[0] for x in w)}:
                if len(ini) >= 2 and ini in keys and ini != k: uf.u(k, ini)
    if FUZZY < 1:
        ks = sorted(k for k in keys if len(k) >= 5 and not re.search(r"\d", k))
        for i, a in enumerate(ks):
            for b in ks[i+1:]:
                if abs(len(a) - len(b)) > 3: continue
                r = difflib.SequenceMatcher(None, a, b).ratio()
                if r >= FUZZY and uf.f(a) != uf.f(b):
                    uf.u(a, b); print("  fuzzy merge [%s]: '%s' ~ '%s' (%.2f)" % (c, a, b, r))
    ufs[c] = uf

nodes, variants, rows_ids = {}, defaultdict(Counter), []
for row in parsed:
    ids = []
    for c in COLS:
        for raw, base, acr in row[c]:
            key = ufs[c].f(norm(base))
            nid = key if MERGE_ACROSS_COLS else c + "::" + key
            nodes.setdefault(nid, {"id": nid, "col": c, "rows": 0})
            variants[nid][raw] += 1
            if nid not in ids: ids.append(nid)
    for nid in ids: nodes[nid]["rows"] += 1
    rows_ids.append(ids)
for nid, n in nodes.items():
    vs = variants[nid]
    n["label"] = max(vs, key=lambda s: (len(s), vs[s]))
    n["aliases"] = sorted(v for v in vs if v != n["label"])

# ───────── GEOMETRY ─────────
def fetch_first(urls):
    for u in urls:
        try: return json.loads(urllib.request.urlopen(u, timeout=40).read().decode("utf-8"))
        except Exception as e: print("  !! could not download", u, "->", e)
    return None
states_gj, world_gj = fetch_first(STATES_URLS), fetch_first(WORLD_URLS)

def _ring(r):
    a = cx = cy = 0.0
    for (x0, y0, *_), (x1, y1, *_) in zip(r, r[1:] + r[:1]):
        c = x0 * y1 - x1 * y0; a += c; cx += (x0 + x1) * c; cy += (y0 + y1) * c
    a /= 2
    return (abs(a), (cx / (6 * a), cy / (6 * a))) if a else (0, (r[0][0], r[0][1]))
def polys(g): return [g["coordinates"]] if g["type"] == "Polygon" else g["coordinates"]
def in_ring(x, y, r):
    ins, j = False, len(r) - 1
    for i in range(len(r)):
        xi, yi, xj, yj = r[i][0], r[i][1], r[j][0], r[j][1]
        if (yi > y) != (yj > y) and x < (xj - xi) * (y - yi) / (yj - yi) + xi: ins = not ins
        j = i
    return ins
def in_poly(x, y, p): return in_ring(x, y, p[0]) and not any(in_ring(x, y, h) for h in p[1:])
def in_geom(x, y, g): return any(in_poly(x, y, p) for p in polys(g))
def seg_d(px, py, ax, ay, bx, by):
    dx, dy = bx - ax, by - ay
    t = 0 if dx == dy == 0 else max(0, min(1, ((px - ax) * dx + (py - ay) * dy) / (dx * dx + dy * dy)))
    return ((px - ax - t * dx) ** 2 + (py - ay - t * dy) ** 2) ** .5
def interior(g):
    best = max(polys(g), key=lambda p: _ring(p[0])[0])
    lon, lat = _ring(best[0])[1]
    if in_poly(lon, lat, best): return lat, lon
    ring = best[0]; xs = [q[0] for q in ring]; ys = [q[1] for q in ring]
    x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys); bd, bp, N = -1, (lat, lon), 26
    for i in range(1, N):
        for j in range(1, N):
            x, y = x0 + (x1 - x0) * i / N, y0 + (y1 - y0) * j / N
            if in_poly(x, y, best):
                d = min(seg_d(x, y, ring[k][0], ring[k][1], ring[k+1][0], ring[k+1][1]) for k in range(len(ring) - 1))
                if d > bd: bd, bp = d, (y, x)
    return bp
def rnd(c): return [round(c[0], 2), round(c[1], 2)] if isinstance(c[0], (int, float)) else [rnd(x) for x in c]

FEAT, cent, BB = {}, {}, {}
for pref, gj in (("S:", states_gj), ("C:", world_gj)):
    for f in (gj or {}).get("features", []):
        nm = f["properties"].get("name")
        if not nm or not f.get("geometry"): continue
        fid = pref + nm
        g = {"type": f["geometry"]["type"], "coordinates": rnd(f["geometry"]["coordinates"])}
        FEAT[fid] = {"type": "Feature", "properties": {"fid": fid, "name": nm}, "geometry": g}
        cent[fid] = interior(g)
        pts = [q for p in polys(g) for q in p[0]]
        BB[fid] = (min(q[0] for q in pts), min(q[1] for q in pts), max(q[0] for q in pts), max(q[1] for q in pts))
def feat_of(lat, lon, pref):
    for fid, (a, b, c, d) in BB.items():
        if fid[0] == pref and a <= lon <= c and b <= lat <= d and in_geom(lon, lat, FEAT[fid]["geometry"]): return fid
    return None

LK = {}
for fid in cent:
    if fid.startswith("C:"): LK[norm(fid[2:])] = ("country", [fid])
for k, vs in COUNTRY_ALIAS.items():
    for v in vs:
        if "C:" + v in cent: LK[norm(k)] = ("country", ["C:" + v]); break
for fid in cent:
    if fid.startswith("S:"): LK[norm(fid[2:])] = ("state", [fid])
for k, v in REGIONS.items():
    mem = ["S:" + n for n in v if "S:" + n in cent]
    if mem: LK[norm(k)] = ("region", mem)

def mk(kind, mem):
    if len(mem) == 1:
        lat, lon = cent[mem[0]]; fid = mem[0]
    else:
        lat = sum(cent[m][0] for m in mem) / len(mem); lon = sum(cent[m][1] for m in mem) / len(mem)
        fid = next((m for m in mem if in_geom(lon, lat, FEAT[m]["geometry"])), None)
        if fid is None:
            fid = min(mem, key=lambda m: (cent[m][0] - lat) ** 2 + ((cent[m][1] - lon) * .78) ** 2)
            lat, lon = cent[fid]
    return {"kind": kind, "members": mem, "lat": round(lat, 4), "lon": round(lon, 4), "fid": fid}
def resolve_one(s):
    k = norm(s)
    if not k: return None, ""
    if k in GLOBAL: return {"kind": "global", "members": ["*"]}, "exact"
    if k in LK: return mk(*LK[k]), "exact"
    st = s.strip()
    if st in ab2name and "S:" + ab2name[st] in cent: return mk("state", ["S:" + ab2name[st]]), "abbr"
    for key in sorted(LK, key=len, reverse=True):
        if len(key) >= 4 and re.search(r"\b" + re.escape(key) + r"\b", k): return mk(*LK[key]), "contains '" + key + "'"
    c = difflib.get_close_matches(k, list(LK), n=1, cutoff=GEO_FUZZY)
    if c: return mk(*LK[c[0]]), "fuzzy '" + c[0] + "'"
    return None, ""

unresolved, soft, ngeo = [], [], 0
for nid, n in nodes.items():
    if n["col"] not in GEO_COLS: continue
    for s in [n["label"]] + n["aliases"]:
        g, how = resolve_one(s)
        if g:
            n["loc"] = g; ngeo += 1
            if how != "exact": soft.append("%s -> %s [%s]" % (s, ",".join(m[2:] for m in g["members"][:4]), how))
            break
    else: unresolved.append(n["label"])
print("geo nodes resolved: %d | unresolved: %d" % (ngeo, len(unresolved)))
if soft: print("non-exact geo matches (verify):\n  " + "\n  ".join(soft))
if unresolved: print("geo UNRESOLVED (add to REGIONS / ALIASES):", unresolved)

# ───────── ORG LOCATIONS: driven by the "<col> Location" field ─────────
COORD = re.compile(r"(-?\d{1,3}(?:\.\d+)?)\s*[,/ ]\s*(-?\d{1,3}\.\d+)")        # lat, lon (needs a decimal on the 2nd number)
STATE_NAMES = {norm(v) for v in ab2name.values()}
def _is_state(p): return p.strip() in ab2name or norm(p) in STATE_NAMES or norm(p) in ("usa", "us", "united states")
def comma_smart(t):
    parts = []
    for p in [x.strip() for x in t.split(",") if x.strip()]:
        if parts and _is_state(p) and not _is_state(parts[-1]): parts[-1] += ", " + p
        else: parts.append(p)
    return parts
def split_locs(cell, n):
    """align a location cell to n orgs. ';' '|' newline preferred; else smart comma ('City, ST' stay together); 1 location -> all."""
    cell = str(cell).strip()
    if not cell or cell.lower() in NA or n == 0: return [None] * n, "empty"
    t = COORD.sub(lambda m: "@@%s|%s@@" % (m[1], m[2]), cell)
    semi = [p.strip() for p in re.split(r"[;\n|]", t) if p.strip()]
    if len(semi) == n: return semi, "ok"
    cs = comma_smart(t)
    if len(cs) == n: return cs, "ok"
    if len(semi) == 1 and len(cs) == 1: return [semi[0]] * n, "shared"
    if n == 1: return [t], "ok"
    return [None] * n, "MISMATCH (%d orgs vs %d/%d locations)" % (n, len(semi), len(cs))

votes, mism = defaultdict(Counter), []
for (_, drow), prow in zip(df.iterrows(), parsed):
    for c, lcn in LOC_FIELD.items():
        orgs = prow[c]; locs, st = split_locs(drow[lcn], len(orgs))
        if st.startswith("MISMATCH"): mism.append("row: %s | %s | '%s'" % ([o[0] for o in orgs][:4], st, str(drow[lcn])[:90]))
        for (raw, base, acr), loc in zip(orgs, locs):
            if loc: votes[(MERGE_ACROSS_COLS and norm(base) or c + "::" + ufs[c].f(norm(base))) if False else (c + "::" + ufs[c].f(norm(base)) if not MERGE_ACROSS_COLS else ufs[c].f(norm(base)))][loc] += 1
if mism:
    print("!! %d rows where the number of End users != number of locations (use ';' between entries to fix):" % len(mism))
    print("  " + "\n  ".join(mism[:25]))

GC = json.load(open(GEOCODE_CACHE)) if os.path.exists(GEOCODE_CACHE) else {}
def nominatim(q):
    if q in GC: return GC[q]
    try:
        req = urllib.request.Request("https://nominatim.openstreetmap.org/search?format=json&limit=1&addressdetails=0&q=" + urllib.parse.quote(q),
                                     headers={"User-Agent": "network-viz-colab/1.0 (research)"})
        res = json.loads(urllib.request.urlopen(req, timeout=20).read().decode()); time.sleep(1.1)
        out = [float(res[0]["lat"]), float(res[0]["lon"]), res[0]["display_name"][:90]] if res else None
    except Exception as e:
        print("  geocode failed:", q, e); out = None
    GC[q] = out; return out

def resolve_loc(s):
    """location text -> (lat, lon, place, how). Order: lat/lon in text > state/country/region name (feature centroid) > Nominatim."""
    m = re.search(r"@@(-?[\d.]+)\|(-?[\d.]+)@@", s)
    if m: return float(m[1]), float(m[2]), s.split("@@")[0].strip() or "coordinates", "csv: coordinates"
    k = norm(s)
    if k in LK:
        g = mk(*LK[k]); return g["lat"], g["lon"], s.strip(), "csv: feature centroid"
    if s.strip() in ab2name and "S:" + ab2name[s.strip()] in cent:
        g = mk("state", ["S:" + ab2name[s.strip()]]); return g["lat"], g["lon"], ab2name[s.strip()], "csv: state centroid"
    if GEOCODE_ONLINE:
        g = nominatim(s)
        if g: return g[0], g[1], g[2], "csv: geocoded"
    return None

org_rows, org_missing = [], []
for nid, n in nodes.items():
    if n["col"] not in ORG_COLS: continue
    vs = [n["label"]] + n["aliases"]; hit = None; txt = ""
    man = {norm(k): v for k, v in LOCATIONS.items()}
    for v in vs:
        if norm(v) in man: txt = "(manual)"; hit = (*man[norm(v)], "manual override"); break
    if not hit and votes.get(nid):
        ranked = votes[nid].most_common()
        txt = ranked[0][0]
        if len(ranked) > 1: print("  note: '%s' has %d different locations; using most common '%s'" % (n["label"][:50], len(ranked), re.sub("@@.*?@@", "coords", txt)))
        hit = resolve_loc(txt)
    if hit:
        lat, lon, place, how = hit
        n["loc"] = {"kind": "point", "lat": round(lat, 4), "lon": round(lon, 4), "place": place[:90], "how": how,
                    "fid": feat_of(lat, lon, "S") or feat_of(lat, lon, "C")}
        org_rows.append({"org": n["label"], "location_text": re.sub("@@.*?@@", "", txt) if txt else "", "resolved": place, "lat": round(lat, 4), "lon": round(lon, 4), "how": how})
    else:
        org_missing.append((n["label"], re.sub("@@.*?@@", "coords", txt) if txt else "(no location in CSV)"))
if GEOCODE_ONLINE: json.dump(GC, open(GEOCODE_CACHE, "w"))
print("\norg locations: %d placed | %d missing" % (len(org_rows), len(org_missing)))
for r in org_rows: print("  %-50s | %-34s -> %s [%s]" % (r["org"][:50], r["location_text"][:34], r["resolved"][:55], r["how"]))
if org_missing:
    print("\nUNPLACED:"); [print("  %-50s | %s" % (a[:50], b)) for a, b in org_missing]
pd.DataFrame(org_rows).to_csv("org_locations_resolved.csv", index=False)

# ───────── payload ─────────
rows = []
for ids in rows_ids:
    geo = sorted({m for nid in ids if nodes[nid].get("loc", {}).get("kind") not in (None, "point") for m in nodes[nid]["loc"]["members"]})
    rows.append({"ids": ids, "geo": geo})
print("\nrows with polygon geography: %d/%d" % (sum(bool(r["geo"]) for r in rows), len(rows)))
used_c = {m for n in nodes.values() if n.get("loc", {}).get("kind") not in (None, "point") for m in n["loc"]["members"] if m.startswith("C:")}
fc = lambda fids: {"type": "FeatureCollection", "features": [FEAT[f] for f in fids if f in FEAT]}
colors = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(COLS)}
cnt = Counter(n["col"] for n in nodes.values())
data = {"cols": COLS, "colors": colors, "counts": {c: cnt[c] for c in COLS}, "hub": HUB_COL, "geoCols": GEO_COLS, "orgCols": ORG_COLS,
        "nodes": list(nodes.values()), "rows": rows,
        "geo": {"states": fc([f for f in FEAT if f.startswith("S:")]), "countries": fc(sorted(used_c))},
        "gibs": {"date": IMAGERY_DATE, "max": date.today().isoformat(), "base": GIBS_BASE, "merged": GIBS_MERGED, "over": GIBS_OVER}}

# ───────── HTML ─────────
HTML = r"""<!doctype html><html><head><meta charset="utf-8"><title>Network v2.3</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<style>
html,body{margin:0;height:100%;background:#0b0e14;color:#dde3ee;font:12px/1.35 system-ui,sans-serif;overflow:hidden}
#graph,#map{position:fixed;inset:0}#map{display:none;z-index:0;background:#0b0e14}
.p{position:fixed;top:8px;background:#141923ee;border:1px solid #2a3142;border-radius:8px;padding:10px;overflow-y:auto;z-index:1000}
#panel{left:8px;width:262px;max-height:calc(100vh - 36px)}
#info{right:8px;width:290px;display:none;max-height:calc(100vh - 120px)}
h4{margin:10px 0 4px;font-size:11px;text-transform:uppercase;letter-spacing:.06em;color:#8a93a6}h4:first-child{margin-top:0}
label.col{display:flex;align-items:center;gap:6px;padding:1px 0;cursor:pointer}
label.col i{width:10px;height:10px;border-radius:50%;display:inline-block}label.col span{margin-left:auto;color:#6b7488}
input[type=range]{width:100%;margin:2px 0}select,input[type=text],input[type=date],button{background:#1d2431;color:#dde3ee;border:1px solid #333c50;border-radius:5px;padding:4px 6px;font-size:12px;color-scheme:dark}
select,input[type=text]{width:100%;box-sizing:border-box}button{cursor:pointer}button:hover{background:#2a3446}button.on{background:#2ec4b6;color:#000;border-color:#2ec4b6}
.row{display:flex;gap:4px;margin:3px 0;align-items:center}.row>*{flex:1}.v{float:right;color:#8a93a6}
#info a{color:#7cc4ff;cursor:pointer;display:block;padding:1px 0}#info a:hover{text-decoration:underline}
#stats{position:fixed;top:8px;left:50%;transform:translateX(-50%);background:#141923cc;padding:4px 10px;border-radius:12px;color:#8a93a6;z-index:1000;white-space:nowrap}
.chip{display:inline-block;background:#2a3446;border:1px solid #ffd166;border-radius:10px;padding:1px 7px;margin:2px 2px 0 0;cursor:pointer}
.leaflet-container{font:inherit}.lbl{background:#000a;color:#fff;border:0;box-shadow:none;padding:1px 4px;font-size:11px}.lbl:before{display:none}
.dia{width:11px;height:11px;margin:1px;transform:rotate(45deg);border:1.5px solid #fff;box-sizing:border-box}
</style></head><body>
<div id="graph"></div><div id="map"></div>
<div id="panel" class="p">
 <div class="row"><button id="vNet" class="on">3D network</button><button id="vMap">Map</button></div>
 <label class="col"><input type="checkbox" id="pin"> pin geography to map (3D view)</label>
 <h4>Search</h4><input type="text" id="q" list="dl" placeholder="node / alias, Enter"><datalist id="dl"></datalist>
 <h4>Geography filter <span class="v"><a href="#" id="gclr">clear</a></span></h4>
 <select id="gadd"><option value="">+ add place…</option></select><div id="chips"></div>
 <div style="color:#6b7488">or click states / countries on the Map</div>
 <h4>Map placement</h4>
 <div>Place non-location nodes (PI, model…) at</div><select id="acol"></select>
 <label class="col"><input type="checkbox" id="cent"> show feature centroid dots</label>
 <label class="col"><input type="checkbox" id="shade" checked> shade geographies by activity</label>
 <div>Network edges (map)</div><select id="edges"><option value="auto">auto</option><option value="all">all</option><option value="sel">selected node only</option><option value="none">none</option></select>
 <h4>Map imagery</h4>
 <div style="color:#6b7488">basemap / overlay picker: bottom-right of map</div>
 <div class="row"><button id="dprev" style="flex:1">◀</button><input type="date" id="gdate" style="flex:4"><button id="dnext" style="flex:1">▶</button></div>
 <div>NASA data-overlay opacity</div><input type="range" id="iop" min="0.1" max="1" step="0.05" value="0.7">
 <h4>Columns <span class="v"><a href="#" id="all">all</a> · <a href="#" id="none">none</a></span></h4><div id="legend"></div>
 <h4>Link mode</h4>
 <select id="mode"><option value="all">All pairs in a row</option><option value="hub">Hub column → everything</option><option value="chain">Chain by column order</option></select>
 <div class="row"><select id="hub"></select></div>
 <label class="col"><input type="checkbox" id="intra" checked> link values within same column</label>
 <h4>Filters</h4>
 <div>Min edge weight <span class="v" id="vW">1</span></div><input type="range" id="minW" min="1" max="20" value="1">
 <div>Min node degree <span class="v" id="vD">0</span></div><input type="range" id="minDeg" min="0" max="30" value="0">
 <h4>Appearance</h4>
 <label class="col"><input type="checkbox" id="labels"> text labels</label>
 <div>Label min degree <span class="v" id="vL">0</span></div><input type="range" id="labelDeg" min="0" max="30" value="0">
 <div>Node size</div><input type="range" id="size" min="0.3" max="4" step="0.1" value="1">
 <div>Link opacity (3D)</div><input type="range" id="op" min="0.05" max="1" step="0.05" value="0.35">
 <label class="col"><input type="checkbox" id="part"> flow particles (3D)</label>
 <label class="col"><input type="checkbox" id="rot"> auto-rotate (3D)</label>
 <h4>Physics (3D)</h4>
 <div>Link distance</div><input type="range" id="dist" min="10" max="300" value="40">
 <div>Repulsion</div><input type="range" id="chg" min="5" max="400" value="30">
 <div class="row"><button id="dim">2D / 3D</button><button id="fit">Fit view</button><button id="snap">PNG</button></div>
 <div style="color:#6b7488;margin-top:6px">◆ = org at its CSV location · ● = everything else · Esc = clear selection</div>
</div>
<div id="info" class="p"></div><div id="stats"></div>
<script src="https://unpkg.com/three@0.150.1/build/three.min.js"></script>
<script src="https://unpkg.com/three-spritetext@1.8.0/dist/three-spritetext.min.js"></script>
<script src="https://unpkg.com/3d-force-graph@1.73.2/dist/3d-force-graph.min.js"></script>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script>
const D=__DATA__, $=id=>document.getElementById(id);
const esc=s=>String(s).replace(/[&<>"]/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));
const NODE=new Map(D.nodes.map(n=>[n.id,n])), cache=new Map(), CUR={links:[],nodes:[],act:[],anch:new Map(),ver:0,lay:null,stat:''};
const LOC=new Map(); for(const n of D.nodes) if(n.loc&&n.loc.lat!=null) LOC.set(n.id,n.loc);
const ORDER=[...D.geoCols,...D.orgCols];
const S={vis:new Set(D.cols),mode:'all',hub:D.hub,minW:1,minDeg:0,intra:true,labels:false,labelDeg:0,size:1,hl:null,sel:null,dim:3,
 view:'net',pin:false,geoSel:new Set(),shade:true,edges:'auto',date:D.gibs.date,iop:.7,anchor:'auto',cent:false};
const sid=x=>typeof x==='object'?x.id:x;
const K=Math.cos(38*Math.PI/180), SC=2.2, proj=(lat,lon)=>[lon*K*SC,lat*SC];
const hsh=s=>{let h=2166136261;for(let i=0;i<s.length;i++){h^=s.charCodeAt(i);h=Math.imul(h,16777619)}return(h>>>0)/4294967295};
const jit=(id,k)=>[(hsh(id+'a')-.5)*k,(hsh(id+'b')-.5)*k];
function passG(r){if(!S.geoSel.size)return true;if(r.geo.includes('*'))return true;return r.geo.some(g=>S.geoSel.has(g))}

function anchors(rows){
 const A=new Map(LOC),tally=new Map(),msum=new Map();
 for(const r of rows){
  const cand=r.ids.filter(i=>LOC.has(i));if(!cand.length)continue;
  let cs=cand;
  if(S.anchor!=='mean'){const order=S.anchor==='auto'?ORDER:[S.anchor,...ORDER];
   for(const c of order){const x=cand.filter(i=>NODE.get(i).col===c);if(x.length){cs=x;break}}}
  for(const id of r.ids){if(LOC.has(id))continue;
   if(S.anchor==='mean'){const s=msum.get(id)||[0,0,0];cs.forEach(c=>{const l=LOC.get(c);s[0]+=l.lat;s[1]+=l.lon;s[2]++});msum.set(id,s)}
   else{const t=tally.get(id)||new Map();cs.forEach(c=>t.set(c,(t.get(c)||0)+1/cs.length));tally.set(id,t)}}}
 for(const [id,t] of tally){let b=null,bv=-1;for(const [c,v] of t)if(v>bv){bv=v;b=c}A.set(id,LOC.get(b))}
 for(const [id,s] of msum)A.set(id,{kind:'mean',lat:s[0]/s[2],lon:s[1]/s[2]});
 return A}
const ANC={};const getAnch=()=>ANC[S.anchor]||(ANC[S.anchor]=anchors(D.rows));

function labelObj(n){
 if(!S.labels||n.deg<S.labelDeg||typeof SpriteText==='undefined')return null;
 const s=new SpriteText(n.label,3.2,D.colors[n.col]);
 s.position.y=Math.cbrt(Math.max(1,Math.sqrt(n.deg+1))*S.size)*4+4;s.material.depthWrite=false;return s}
function nodeTip(n){const pl=n.loc&&n.loc.place?'<br>📍 '+esc(n.loc.place):'';
 return `<div style="background:#000d;padding:4px 8px;border-radius:4px;font:12px system-ui"><b>${esc(n.label)}</b><br>${esc(n.col)} · ${n.deg} links · ${n.rows} rows${pl}${n.aliases.length?'<br><i>aka '+esc(n.aliases.join(', '))+'</i>':''}</div>`}
const G=ForceGraph3D({controlType:'trackball'})($('graph')).backgroundColor('#0b0e14').nodeRelSize(4)
 .nodeVal(n=>Math.max(1,Math.sqrt(n.deg+1))*S.size)
 .nodeColor(n=>S.hl&&!S.hl.n.has(n.id)?'#262b36':D.colors[n.col])
 .nodeLabel(nodeTip)
 .linkColor(l=>S.hl&&!S.hl.l.has(l)?'#1b1f29':'#8a93a6').linkOpacity(.35)
 .linkWidth(l=>S.hl&&S.hl.l.has(l)?2:Math.min(.3+l.w*.35,4)).linkDirectionalParticleWidth(1.6)
 .nodeThreeObjectExtend(true).nodeThreeObject(labelObj).onNodeClick(pick).onBackgroundClick(()=>select(null));
const anchorForce=()=>{let ns=[];const f=a=>{if(!S.pin)return;for(const n of ns){if(n.ax==null||LOC.has(n.id))continue;
 n.vx+=(n.ax-n.x)*.3*a;n.vy+=(n.ay-n.y)*.3*a;if(S.dim===3)n.vz+=(0-n.z)*.03*a}};f.initialize=x=>{ns=x};return f};
G.d3Force('anchor',anchorForce());
let fitted=false;G.onEngineStop(()=>{if(!fitted){fitted=true;G.zoomToFit(600,60)}});
const refresh=()=>{if(S.view==='map'){renderMap();return}G.nodeColor(G.nodeColor());G.linkColor(G.linkColor());G.linkWidth(G.linkWidth())};

let OUT=null;
function outlines(on){
 const sc=G.scene();if(OUT){sc.remove(OUT);OUT.geometry.dispose();OUT=null}if(!on)return;
 const pts=[],ring=r=>{for(let i=0;i<r.length-1;i++){if(Math.abs(r[i][0]-r[i+1][0])>180)continue;
  const a=proj(r[i][1],r[i][0]),b=proj(r[i+1][1],r[i+1][0]);pts.push(a[0],a[1],0,b[0],b[1],0)}};
 for(const fc of [D.geo.states,D.geo.countries])for(const f of fc.features){const g=f.geometry;(g.type==='Polygon'?[g.coordinates]:g.coordinates).forEach(p=>p.forEach(ring))}
 const geo=new THREE.BufferGeometry();geo.setAttribute('position',new THREE.Float32BufferAttribute(pts,3));
 OUT=new THREE.LineSegments(geo,new THREE.LineBasicMaterial({color:0x3a4a6b,transparent:true,opacity:.75}));sc.add(OUT)}
function applyPin(){
 for(const n of CUR.nodes){const a=CUR.anch.get(n.id);
  if(S.pin&&a){const p=proj(a.lat,a.lon);n.ax=p[0];n.ay=p[1];
   if(LOC.has(n.id)){const j=jit(n.id,6);n.fx=p[0]+j[0];n.fy=p[1]+j[1];n.fz=0}
   else{n.fx=n.fy=n.fz=null;if(n.x==null){n.x=p[0]+Math.random()*4;n.y=p[1]+Math.random()*4;n.z=Math.random()*10}}}
  else{n.fx=n.fy=n.fz=null;n.ax=n.ay=null}}}

function build(){
 const rows=CUR.act=D.rows.filter(passG);
 const W=new Map(),cnt=new Map(),add=(a,b)=>{if(a===b)return;const k=a<b?a+'\u0001'+b:b+'\u0001'+a;W.set(k,(W.get(k)||0)+1)};
 const clique=L=>{for(let i=0;i<L.length;i++)for(let j=i+1;j<L.length;j++)add(L[i],L[j])};
 for(const row of rows){
  const ids=row.ids.filter(i=>S.vis.has(NODE.get(i).col)),col=i=>NODE.get(i).col;
  ids.forEach(i=>cnt.set(i,(cnt.get(i)||0)+1));
  if(S.mode==='all'){for(let i=0;i<ids.length;i++)for(let j=i+1;j<ids.length;j++){if(!S.intra&&col(ids[i])===col(ids[j]))continue;add(ids[i],ids[j])}}
  else if(S.mode==='hub'){const h=ids.filter(i=>col(i)===S.hub),r=ids.filter(i=>col(i)!==S.hub);h.forEach(a=>r.forEach(b=>add(a,b)))}
  else{const by={};ids.forEach(i=>(by[col(i)]??=[]).push(i));const cs=D.cols.filter(c=>by[c]);
   for(let k=0;k<cs.length-1;k++)by[cs[k]].forEach(a=>by[cs[k+1]].forEach(b=>add(a,b)));if(S.intra)cs.forEach(c=>clique(by[c]))}}
 const raw=[],deg=new Map();
 for(const [k,w] of W){if(w<S.minW)continue;const [a,b]=k.split('\u0001');raw.push({source:a,target:b,w});deg.set(a,(deg.get(a)||0)+1);deg.set(b,(deg.get(b)||0)+1)}
 const nodes=D.nodes.filter(n=>S.vis.has(n.col)&&cnt.has(n.id)&&(deg.get(n.id)||0)>=S.minDeg).map(n=>{
  let o=cache.get(n.id);if(!o){o={...n};cache.set(n.id,o)}o.deg=deg.get(n.id)||0;o.rows=cnt.get(n.id);return o});
 const keep=new Set(nodes.map(n=>n.id)),links=raw.filter(l=>keep.has(l.source)&&keep.has(l.target));
 CUR.links=links;CUR.nodes=nodes;CUR.ver++;CUR.anch=getAnch();applyPin();
 G.graphData({nodes,links});G.nodeThreeObject(labelObj);
 CUR.stat=`${nodes.length} nodes · ${links.length} edges · ${rows.length}/${D.rows.length} rows`;$('stats').textContent=CUR.stat;
 const s=S.sel&&cache.get(S.sel);if(s&&keep.has(s.id))select(s);else select(null)}
function select(n){
 S.sel=n?n.id:null;
 if(!n){S.hl=null;$('info').style.display='none';refresh();return}
 const ns=new Set([n.id]),ls=new Set(),by={};
 for(const l of CUR.links){const a=sid(l.source),b=sid(l.target);
  if(a===n.id||b===n.id){ls.add(l);const o=a===n.id?b:a;ns.add(o);(by[NODE.get(o).col]??=[]).push([o,l.w])}}
 S.hl={n:ns,l:ls};
 let h=`<h4>${esc(n.col)}</h4><b style="font-size:14px;color:${D.colors[n.col]}">${esc(n.label)}</b>`;
 if(n.aliases.length)h+=`<div><i>aka ${esc(n.aliases.join(', '))}</i></div>`;
 h+=`<div>${n.rows} rows · ${ns.size-1} connections</div>`;
 if(n.loc){const l=n.loc;h+=`<div>${l.kind==='point'?'📍 '+esc(l.place)+' <i>('+esc(l.how)+')</i>':l.kind==='global'?'global':'geo: '+esc(l.members.length>6?l.members.length+' areas':l.members.map(m=>m.slice(2)).join(', '))}</div>`}
 for(const c of D.cols){const L=by[c];if(!L)continue;L.sort((a,b)=>b[1]-a[1]);
  h+=`<h4><span style="color:${D.colors[c]}">●</span> ${esc(c)} (${L.length})</h4>`+L.map(([o,w])=>`<a data-id="${esc(o)}">${esc(NODE.get(o).label)} <span style="color:#6b7488">×${w}</span></a>`).join('')}
 $('info').innerHTML=h;$('info').style.display='block';refresh()}
function pick(n){select(n);if(!n)return;
 if(S.view==='map'){const p=CUR.lay&&CUR.lay.P.get(n.id);if(p)M.flyTo(p,Math.max(M.getZoom(),5));return}
 const x=n.x||0,y=n.y||0,z=n.z||0,d=Math.hypot(x,y,z)||1,r=1+120/d;G.cameraPosition({x:x*r,y:y*r,z:z*r},{x,y,z},800)}
$('info').onclick=e=>{const o=cache.get(e.target.dataset.id);if(o)pick(o)};
addEventListener('keydown',e=>{if(e.key==='Escape')select(null)});

let M=null,GL=null,NL=null,LC=null;const GF={},TL=[],OVR=[];
const BLANK='data:image/gif;base64,R0lGODlhAQABAAAAACH5BAEKAAEALAAAAAABAAEAAAICTAEAOw==';
function gibsUrl(g){const t=g[4]==='date'?S.date:g[4];
 return `https://gibs.earthdata.nasa.gov/wmts/epsg3857/best/${g[1]}/default/${t?t+'/':''}${g[2]}/{z}/{y}/{x}.${g[3]}`}
function gibsLayer(g,z,over){
 const mz=+g[2].match(/Level(\d+)/)[1];
 const l=L.tileLayer(gibsUrl(g),{maxNativeZoom:mz,maxZoom:19,tileSize:256,noWrap:true,bounds:[[-85.05,-180],[85.05,180]],errorTileUrl:BLANK,
  attribution:'Imagery: NASA GIBS / Worldview',zIndex:z,opacity:over&&g[5]?S.iop:1});
 l._g=g;TL.push(l);if(over&&g[5])OVR.push(l);return l}
const esri=(p,nz,z)=>L.tileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/'+p+'/MapServer/tile/{z}/{y}/{x}',
 {maxNativeZoom:nz||18,maxZoom:19,noWrap:true,zIndex:z||1,errorTileUrl:BLANK,attribution:'Tiles © Esri — Esri, Maxar, Earthstar Geographics, USGS, NOAA'});
function initMap(){
 if(M)return;
 M=L.map('map',{preferCanvas:true,minZoom:2,maxZoom:13,zoomSnap:1,maxBounds:[[-85,-180],[85,180]],maxBoundsViscosity:1}).setView([39,-96],4);
 const base={},over={};
 const dflt=esri('World_Imagery',18,1);base['Esri satellite (full coverage)']=dflt;dflt.addTo(M);
 base['Esri satellite + place labels']=L.layerGroup([esri('World_Imagery',18,1),esri('Reference/World_Boundaries_and_Places',18,3)]);
 base['Esri topographic']=esri('World_Topo_Map',18,1);
 base['Esri street']=esri('World_Street_Map',18,1);
 base['Esri light gray']=esri('Canvas/World_Light_Gray_Base',16,1);
 base['Esri dark gray']=esri('Canvas/World_Dark_Gray_Base',16,1);
 base['OpenStreetMap']=L.tileLayer('https://tile.openstreetmap.org/{z}/{x}/{y}.png',{maxZoom:19,noWrap:true,errorTileUrl:BLANK,attribution:'© OpenStreetMap contributors'});
 const byId=Object.fromEntries(D.gibs.base.map(g=>[g[1],g]));
 D.gibs.merged.forEach(m=>{base[m[0]]=L.layerGroup(m[1].map((id,i)=>gibsLayer(byId[id],1+i,false)))});
 D.gibs.base.forEach(g=>{base[g[0]]=gibsLayer(g,1,false)});
 base['No basemap']=L.layerGroup();
 D.gibs.over.forEach(g=>{const l=gibsLayer(g,5,true);over[g[0]]=l;if(g[6])l.addTo(M)});
 LC=L.control.layers(base,over,{position:'bottomright',collapsed:true}).addTo(M);
 L.control.scale({position:'bottomleft'}).addTo(M);
 GL=L.layerGroup().addTo(M);
 for(const fc of [D.geo.states,D.geo.countries])
  L.geoJSON(fc,{style:()=>({weight:.8,color:'#ffffff',opacity:.7,fillOpacity:0}),onEachFeature:(f,l)=>{GF[f.properties.fid]=l;
   l.on('click',()=>toggleGeo(f.properties.fid));l.bindTooltip('',{sticky:true})}}).addTo(GL);
 NL=L.layerGroup().addTo(M);
 M.on('zoomend',()=>{if(S.view==='map')renderMap()})}
function updateChoro(){
 const c={};let mx=1;
 for(const r of CUR.act)for(const g of r.geo){if(g==='*')continue;c[g]=(c[g]||0)+1;if(c[g]>mx)mx=c[g]}
 const foot=new Set();
 if(S.sel){const n=NODE.get(S.sel);if(n&&n.loc&&n.loc.kind!=='point'&&n.loc.members)n.loc.members.forEach(m=>foot.add(m));
  for(const r of CUR.act)if(r.ids.includes(S.sel))r.geo.forEach(g=>foot.add(g))}
 for(const [fid,l] of Object.entries(GF)){const v=c[fid]||0,sel=S.geoSel.has(fid),f=foot.has(fid);
  l.setStyle({fillColor:f?'#ffd166':'#2ec4b6',fillOpacity:f?.35:(S.shade&&v?.06+.38*v/mx:0),weight:sel?3:f?1.8:.8,
   color:(sel||f)?'#ffd166':'#ffffff',opacity:(sel||f)?1:.7});
  l.setTooltipContent(`<b>${esc(l.feature.properties.name)}</b><br>${v} rows`+(sel?'<br>(geo filter on)':'<br>click to filter'))}}
function layout(){
 const key=CUR.ver+'|'+M.getZoom();if(CUR.lay&&CUR.lay.key===key)return CUR.lay;
 const grp=new Map();
 for(const n of CUR.nodes){const a=CUR.anch.get(n.id);if(!a)continue;const k=a.lat.toFixed(3)+','+a.lon.toFixed(3);
  if(!grp.has(k))grp.set(k,{a,items:[]});grp.get(k).items.push(n)}
 const P=new Map(),pts=[],cen=[];
 for(const g of grp.values()){
  const it=g.items,a=g.a;
  it.sort((x,y)=>(LOC.has(y.id)-LOC.has(x.id))||x.col.localeCompare(y.col)||x.label.localeCompare(y.label));
  const c=M.latLngToLayerPoint([a.lat,a.lon]);let R=Infinity;
  if(a.kind!=='point'&&a.fid&&GF[a.fid]){const b=GF[a.fid].getBounds(),p1=M.latLngToLayerPoint(b.getSouthWest()),p2=M.latLngToLayerPoint(b.getNorthEast());
   R=.38*Math.min(Math.abs(p2.x-p1.x),Math.abs(p1.y-p2.y))}
  const s=Math.max(3.5,Math.min(12,R/Math.sqrt(Math.max(1,it.length))));
  it.forEach((n,i)=>{const r=s*Math.sqrt(i),ang=i*2.399963,ll=M.layerPointToLatLng(L.point(c.x+r*Math.cos(ang),c.y+r*Math.sin(ang)));P.set(n.id,[ll.lat,ll.lng]);pts.push(n)});
  cen.push([a.lat,a.lon])}
 return CUR.lay={key,P,pts,cen}}
function renderMap(){
 initMap();NL.clearLayers();updateChoro();
 const {P,pts,cen}=layout(),em=S.edges==='auto'?(CUR.links.length>2500?'sel':'all'):S.edges;
 if(em!=='none')for(const l of CUR.links){const a=P.get(sid(l.source)),b=P.get(sid(l.target));if(!a||!b)continue;
  const on=S.hl&&S.hl.l.has(l);if(em==='sel'&&!on)continue;
  L.polyline([a,b],{weight:on?2.4:Math.min(.5+l.w*.3,3),color:on?'#ffd166':'#ffffff',opacity:on?.95:(S.hl?.07:.28),interactive:false}).addTo(NL)}
 if(S.cent)cen.forEach(c=>L.circleMarker(c,{radius:2.5,color:'#fff',weight:1,fillColor:'#000',fillOpacity:1,interactive:false}).addTo(NL));
 let nl=0;
 for(const n of pts){const dim=S.hl&&!S.hl.n.has(n.id),isPt=n.loc&&n.loc.kind==='point',fill=dim?'#555':D.colors[n.col];let m;
  if(isPt)m=L.marker(P.get(n.id),{icon:L.divIcon({className:'',html:`<div class="dia" style="background:${fill};opacity:${dim?.35:1}"></div>`,iconSize:[14,14],iconAnchor:[7,7]})});
  else m=L.circleMarker(P.get(n.id),{radius:4+Math.sqrt(n.deg+1)*S.size*1.3,color:'#fff',weight:1,opacity:dim?.25:.9,fillColor:fill,fillOpacity:dim?.35:.95});
  if(S.labels&&n.deg>=S.labelDeg&&!dim&&nl<250){nl++;m.bindTooltip(esc(n.label),{permanent:true,direction:'right',offset:[isPt?9:6,0],className:'lbl'})}
  else m.bindTooltip(nodeTip(n));
  m.on('click',e=>{L.DomEvent.stopPropagation(e);select(n)});m.addTo(NL)}
 const miss=CUR.nodes.length-pts.length;
 $('stats').textContent=CUR.stat+(miss?` · ${miss} nodes without a location hidden`:'')}
function fitMap(){const {P}=layout(),v=[...P.values()];if(v.length)M.fitBounds(L.latLngBounds(v),{padding:[60,60],maxZoom:7})}
function setView(v){
 S.view=v;$('graph').style.display=v==='net'?'block':'none';$('map').style.display=v==='map'?'block':'none';
 $('vNet').classList.toggle('on',v==='net');$('vMap').classList.toggle('on',v==='map');
 if(v==='map'){G.pauseAnimation();initMap();M.invalidateSize();renderMap();setTimeout(()=>{M.invalidateSize();if(!M._fitted&&CUR.lay&&CUR.lay.pts.length){M._fitted=1;fitMap()}},120)}
 else{G.resumeAnimation();G.width(innerWidth).height(innerHeight);$('stats').textContent=CUR.stat}}
addEventListener('resize',()=>{if(M)M.invalidateSize()});

const setDate=d=>{S.date=d;$('gdate').value=d;TL.filter(l=>l._g[4]==='date').forEach(l=>l.setUrl(gibsUrl(l._g)))};
$('gdate').value=D.gibs.date;$('gdate').max=D.gibs.max;
$('gdate').onchange=e=>{if(e.target.value)setDate(e.target.value)};
const shiftDay=k=>{const d=new Date(S.date+'T00:00:00Z');d.setUTCDate(d.getUTCDate()+k);const s=d.toISOString().slice(0,10);if(s<=D.gibs.max)setDate(s)};
$('dprev').onclick=()=>shiftDay(-1);$('dnext').onclick=()=>shiftDay(1);
$('iop').oninput=e=>{S.iop=+e.target.value;OVR.forEach(l=>l.setOpacity(S.iop))};
$('shade').onchange=e=>{S.shade=e.target.checked;if(S.view==='map')renderMap()};
$('edges').onchange=e=>{S.edges=e.target.value;if(S.view==='map')renderMap()};
$('cent').onchange=e=>{S.cent=e.target.checked;if(S.view==='map')renderMap()};
[['auto','first location column (State → AOI → End users)'],...ORDER.map(c=>[c,c]),['mean','mean of all locations in row']].forEach(([v,t])=>{
 const o=document.createElement('option');o.value=v;o.textContent=t;$('acol').append(o)});
$('acol').onchange=e=>{S.anchor=e.target.value;build()};

const gname=fid=>(D.geo.states.features.concat(D.geo.countries.features).find(f=>f.properties.fid===fid)||{properties:{name:fid}}).properties.name;
function renderChips(){$('chips').innerHTML=[...S.geoSel].map(f=>`<span class="chip" data-f="${esc(f)}">${esc(gname(f))} ×</span>`).join('')}
function toggleGeo(f){S.geoSel.has(f)?S.geoSel.delete(f):S.geoSel.add(f);renderChips();build()}
for(const fc of [D.geo.states,D.geo.countries])
 [...fc.features].sort((a,b)=>a.properties.name.localeCompare(b.properties.name)).forEach(f=>{const o=document.createElement('option');
  o.value=f.properties.fid;o.textContent=f.properties.name+(f.properties.fid.startsWith('C:')?' (country)':'');$('gadd').append(o)});
$('gadd').onchange=e=>{if(e.target.value&&!S.geoSel.has(e.target.value))toggleGeo(e.target.value);e.target.value=''};
$('chips').onclick=e=>{if(e.target.dataset.f)toggleGeo(e.target.dataset.f)};
$('gclr').onclick=e=>{e.preventDefault();S.geoSel.clear();renderChips();build()};

D.cols.forEach(c=>{const l=document.createElement('label');l.className='col';
 l.innerHTML=`<input type="checkbox" checked data-c="${esc(c)}"><i style="background:${D.colors[c]}"></i>${esc(c)}<span>${D.counts[c]}</span>`;$('legend').append(l);
 const o=document.createElement('option');o.value=o.textContent=c;if(c===D.hub)o.selected=true;$('hub').append(o)});
D.nodes.forEach(n=>{const o=document.createElement('option');o.value=n.label;$('dl').append(o)});
$('legend').onchange=e=>{const c=e.target.dataset.c;e.target.checked?S.vis.add(c):S.vis.delete(c);build()};
const setAll=v=>{document.querySelectorAll('#legend input').forEach(i=>{i.checked=v;v?S.vis.add(i.dataset.c):S.vis.delete(i.dataset.c)});build()};
$('all').onclick=e=>{e.preventDefault();setAll(true)};$('none').onclick=e=>{e.preventDefault();setAll(false)};
$('mode').onchange=e=>{S.mode=e.target.value;build()};$('hub').onchange=e=>{S.hub=e.target.value;build()};
$('intra').onchange=e=>{S.intra=e.target.checked;build()};
$('minW').oninput=e=>{S.minW=+e.target.value;$('vW').textContent=S.minW;build()};
$('minDeg').oninput=e=>{S.minDeg=+e.target.value;$('vD').textContent=S.minDeg;build()};
$('labels').onchange=e=>{S.labels=e.target.checked;G.nodeThreeObject(labelObj);if(S.view==='map')renderMap()};
$('labelDeg').oninput=e=>{S.labelDeg=+e.target.value;$('vL').textContent=S.labelDeg;G.nodeThreeObject(labelObj);if(S.view==='map')renderMap()};
$('size').oninput=e=>{S.size=+e.target.value;G.nodeVal(G.nodeVal());G.nodeThreeObject(labelObj);if(S.view==='map')renderMap()};
$('op').oninput=e=>G.linkOpacity(+e.target.value);
$('part').onchange=e=>G.linkDirectionalParticles(e.target.checked?2:0);
$('dist').oninput=e=>{G.d3Force('link').distance(+e.target.value);G.d3ReheatSimulation()};
$('chg').oninput=e=>{G.d3Force('charge').strength(-+e.target.value);G.d3ReheatSimulation()};
let rot=null;$('rot').onchange=e=>{clearInterval(rot);if(e.target.checked)rot=setInterval(()=>{
 const p=G.cameraPosition(),c=Math.cos(.004),s=Math.sin(.004);G.cameraPosition({x:p.x*c+p.z*s,y:p.y,z:-p.x*s+p.z*c})},30)};
$('dim').onclick=()=>{S.dim=S.dim===3?2:3;G.numDimensions(S.dim);G.d3ReheatSimulation()};
$('fit').onclick=()=>S.view==='map'?fitMap():G.zoomToFit(600,60);
$('snap').onclick=()=>{if(S.view==='map'){alert('PNG export is 3D-only; screenshot the map.');return}
 const r=G.renderer();r.render(G.scene(),G.camera());const a=document.createElement('a');a.href=r.domElement.toDataURL('image/png');a.download='network.png';a.click()};
$('pin').onchange=e=>{S.pin=e.target.checked;outlines(S.pin);applyPin();G.d3ReheatSimulation();setTimeout(()=>G.zoomToFit(800,60),1500)};
$('vNet').onclick=()=>setView('net');$('vMap').onclick=()=>setView('map');
$('q').addEventListener('change',e=>{
 const q=e.target.value.trim().toLowerCase();if(!q)return;
 const n=D.nodes.find(x=>x.label.toLowerCase()===q)||D.nodes.find(x=>(x.label+' '+x.aliases.join(' ')).toLowerCase().includes(q));
 if(!n){$('stats').textContent='no match';return}
 if(!S.vis.has(n.col)){S.vis.add(n.col);document.querySelector(`#legend input[data-c="${CSS.escape(n.col)}"]`).checked=true;build()}
 setTimeout(()=>{const o=cache.get(n.id);CUR.nodes.includes(o)?pick(o):$('stats').textContent='node hidden by filters'},300)});
G.d3Force('link').distance(40);G.d3Force('charge').strength(-30);
build();
</script></body></html>"""

html = HTML.replace("__DATA__", json.dumps(data, ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/"))
open(OUT_HTML, "w", encoding="utf-8").write(html)
print("✔ %s: %d nodes, %d rows, %.0f KB" % (OUT_HTML, len(nodes), len(rows), len(html) / 1024))
try:
    from google.colab import files
    files.download(OUT_HTML); files.download("org_locations_resolved.csv")
except Exception: pass

Saving Network_Viz_A9.csv to Network_Viz_A9 (3).csv
15 rows | sep = COMMA | geo cols: ['State', 'Area of Interest', 'Area of Interest_2'] | org cols: ['End users'] | location fields: {'End users': 'End users Location'}
  fuzzy merge [Area of Interest_2]: 'california' ~ 'califronia' (0.90)
  fuzzy merge [End users]: 'us forest service' ~ 'usda forest service' (0.94)
geo nodes resolved: 60 | unresolved: 0
non-exact geo matches (verify):
  Latvia and Sweden -> Latvia [contains 'latvia']
!! 7 rows where the number of End users != number of locations (use ';' between entries to fix):
  row: ['Con Edison', 'National Grid', 'Pacific Gas and Electric Company (PG&E)', 'Southern Company'] | MISMATCH (4 orgs vs 1/7 locations) | 'New York, NY, USA, London, UK, Oakland, CA, USA, Atlanta, GA, USA'
  row: ['Ariel Re', 'Arch', 'Everest'] | MISMATCH (3 orgs vs 1/4 locations) | 'Bermuda, Jersey City, NJ, USA, Hamilton, Bermuda'
  row: ['Sonoma Technology', 'US EPA', "California's South Coast Air Quality

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
# # ── Network Viz v2.1: network + geography on NASA GIBS (Worldview) imagery. No API keys. ──
# import pandas as pd, re, json, difflib, io, urllib.request
# from collections import Counter, defaultdict
# from datetime import date, timedelta

# # ───────── CONFIG ─────────
# CSV_FILE   = None                       # None = upload dialog; or path / raw GitHub URL
# OUT_HTML   = "network_v2.html"
# HUB_COL    = "PI"
# GEO_COLS   = ["State", "Area of Interest", "Area of Interest_2"]
# MERGE_ACROSS_COLS = False
# FUZZY      = 0.88                       # near-dup merge inside a column (1.0 = off)
# GEO_FUZZY  = 0.80                       # typo tolerance for place names
# ALIASES    = {}                         # {"GOES-R": "GOES-ABI", ...}
# IMAGERY_DATE = (date.today() - timedelta(days=2)).isoformat()   # default date for daily GIBS layers

# # NASA GIBS layers. Any Worldview layer ID works: paste its identifier + tile matrix set.
# # (title, layer id, tile matrix set, ext, time)   time: None = static | "date" = follows date picker | "YYYY-MM-DD" = fixed
# GIBS_BASE = [
#     ("Blue Marble: relief + bathymetry", "BlueMarble_ShadedRelief_Bathymetry", "GoogleMapsCompatible_Level8", "jpeg", None),
#     ("Blue Marble: Next Generation",     "BlueMarble_NextGeneration",          "GoogleMapsCompatible_Level8", "jpeg", None),
#     ("MODIS Terra true color (daily)",   "MODIS_Terra_CorrectedReflectance_TrueColor", "GoogleMapsCompatible_Level9", "jpeg", "date"),
#     ("VIIRS SNPP true color (daily)",    "VIIRS_SNPP_CorrectedReflectance_TrueColor",  "GoogleMapsCompatible_Level9", "jpeg", "date"),
#     ("Black Marble night lights (2016)", "VIIRS_Black_Marble",                 "GoogleMapsCompatible_Level8", "png",  "2016-01-01"),
# ]
# # (title, layer id, tile matrix set, ext, time, is_data_layer(opacity slider), on_by_default)
# GIBS_OVER = [
#     ("Borders & features (reference)", "Reference_Features_15m", "GoogleMapsCompatible_Level13", "png", None, False, True),
#     ("Coastlines (reference)",         "Coastlines_15m",         "GoogleMapsCompatible_Level13", "png", None, False, False),
#     ("Place labels (reference)",       "Reference_Labels_15m",   "GoogleMapsCompatible_Level13", "png", None, False, False),
#     ("Land surface temp, day",         "MODIS_Terra_Land_Surface_Temp_Day", "GoogleMapsCompatible_Level7", "png", "date", True, False),
#     ("Aerosol optical depth",          "MODIS_Combined_Value_Added_AOD",    "GoogleMapsCompatible_Level6", "png", "date", True, False),
#     ("Snow cover",                     "MODIS_Terra_NDSI_Snow_Cover",       "GoogleMapsCompatible_Level8", "png", "date", True, False),
# ]
# STATES_URLS = ["https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json",
#                "https://raw.githubusercontent.com/python-visualization/folium/main/examples/data/us-states.json"]
# WORLD_URLS  = ["https://raw.githubusercontent.com/johan/world.geo.json/master/countries.geo.json",
#                "https://raw.githubusercontent.com/python-visualization/folium/main/examples/data/world-countries.json"]
# NA = {"", "nan", "none", "n/a", "na", "-", "tbd", "null"}
# PALETTE = ["#e63946","#f4a261","#2a9d8f","#4361ee","#e9c46a","#9b5de5","#00bbf9","#f15bb5","#80ed99","#ff9f1c",
#            "#06d6a0","#c77dff","#ef476f","#118ab2","#ffd166","#8ac926","#ff595e","#6a994e","#bc6c25","#48cae4"]

# # ───────── GAZETTEER (edit REGIONS to change what a region covers) ─────────
# ABBR = ("AL Alabama,AK Alaska,AZ Arizona,AR Arkansas,CA California,CO Colorado,CT Connecticut,DE Delaware,DC District of Columbia,"
#         "FL Florida,GA Georgia,HI Hawaii,ID Idaho,IL Illinois,IN Indiana,IA Iowa,KS Kansas,KY Kentucky,LA Louisiana,ME Maine,"
#         "MD Maryland,MA Massachusetts,MI Michigan,MN Minnesota,MS Mississippi,MO Missouri,MT Montana,NE Nebraska,NV Nevada,"
#         "NH New Hampshire,NJ New Jersey,NM New Mexico,NY New York,NC North Carolina,ND North Dakota,OH Ohio,OK Oklahoma,OR Oregon,"
#         "PA Pennsylvania,PR Puerto Rico,RI Rhode Island,SC South Carolina,SD South Dakota,TN Tennessee,TX Texas,UT Utah,VT Vermont,"
#         "VA Virginia,WA Washington,WV West Virginia,WI Wisconsin,WY Wyoming")
# ab2name = dict(x.split(" ", 1) for x in ABBR.split(","))
# R = lambda s: [ab2name[a] for a in s.split()]
# ALL48 = [n for a, n in ab2name.items() if a not in ("AK", "HI", "PR")]
# EAST_COAST = R("ME NH MA RI CT NY NJ DE MD VA NC SC GA FL")
# GULF = R("TX LA MS AL FL")
# MIDWEST = R("IL IN IA KS MI MN MO NE ND OH SD WI")
# REGIONS = {
#     "conus": ALL48, "contiguous us": ALL48, "contiguous united states": ALL48, "continental us": ALL48,
#     "continental united states": ALL48, "lower 48": ALL48,
#     "united states": ALL48 + ["Alaska", "Hawaii"], "us": ALL48 + ["Alaska", "Hawaii"], "usa": ALL48 + ["Alaska", "Hawaii"],
#     "conus coastal": R("ME NH MA RI CT NY NJ DE MD VA NC SC GA FL AL MS LA TX CA OR WA"),
#     "us east": R("ME NH VT MA RI CT NY NJ PA DE MD DC VA WV NC SC GA FL OH MI IN KY TN AL MS"),
#     "east coast": EAST_COAST, "us east coast": EAST_COAST, "eastern seaboard": EAST_COAST,
#     "west coast": R("CA OR WA"), "us west": R("WA OR CA NV ID UT AZ MT WY CO NM"),
#     "gulf coast": GULF, "gulf coasts": GULF, "gulf states": GULF,
#     "midwest": MIDWEST, "midwest united states": MIDWEST, "us midwest": MIDWEST, "midwestern us": MIDWEST,
#     "northeast": R("ME NH VT MA RI CT NY NJ PA"), "southeast": R("AL FL GA KY MS NC SC TN VA WV"),
#     "southwest": R("AZ NM TX OK"), "pacific northwest": R("WA OR ID"), "great lakes": R("MN WI IL IN MI OH PA NY"),
#     "great plains": R("ND SD NE KS OK"),
# }
# GLOBAL = {"global", "worldwide", "world"}
# COUNTRY_ALIAS = {"tanzania": ["United Republic of Tanzania", "Tanzania"], "uk": ["United Kingdom"],
#                  "great britain": ["United Kingdom"], "ivory coast": ["Ivory Coast", "Côte d'Ivoire"]}

# # ───────── LOAD ─────────
# if CSV_FILE is None:
#     from google.colab import files
#     up = files.upload(); CSV_FILE = next(iter(up)); raw_bytes = up[CSV_FILE]
# elif CSV_FILE.startswith("http"):
#     raw_bytes = urllib.request.urlopen(CSV_FILE, timeout=30).read()
# else:
#     raw_bytes = open(CSV_FILE, "rb").read()
# try:    text = raw_bytes.decode("utf-8-sig")
# except UnicodeDecodeError: text = raw_bytes.decode("latin-1")
# sep = "\t" if "\t" in text.split("\n", 1)[0] else ","
# df = pd.read_csv(io.StringIO(text), sep=sep, dtype=str, keep_default_na=False)
# df.columns = [c.strip() for c in df.columns]
# COLS = list(df.columns)
# lc = {c.lower(): c for c in COLS}
# GEO_COLS = [lc[g.lower()] for g in GEO_COLS if g.lower() in lc]
# HUB_COL = lc.get(HUB_COL.lower(), COLS[0])
# print(len(df), "rows | sep =", "TAB" if sep == "\t" else "COMMA", "| geo cols:", GEO_COLS)

# # ───────── PARSE / NORMALIZE ─────────
# STOP = {"of", "and", "the", "for", "in", "on", "a", "an", "&"}
# norm = lambda s: re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()
# ALIAS_N = {norm(k): v for k, v in ALIASES.items()}

# def split_cell(v):
#     out = []
#     for p in re.split(r"[;\n|]|,(?![^()]*\))", str(v).strip()):
#         p = re.sub(r"^\s*(and|&)\s+", "", p.strip(), flags=re.I).strip(" ,.;:")
#         while p.endswith(")") and p.count(")") > p.count("("): p = p[:-1].strip()
#         while p.startswith("(") and p.count("(") > p.count(")"): p = p[1:].strip()
#         if p.lower() in NA: continue
#         out.append(ALIAS_N.get(norm(p), p))
#     return out

# def parse(p):
#     m = re.match(r"^(.*?)\s*\(([^()]+)\)\s*$", p)
#     if m and m.group(1) and re.fullmatch(r"[A-Za-z0-9&\-]{2,10}", m.group(2)) and sum(ch.isupper() for ch in m.group(2)) >= 2:
#         return m.group(1).strip(), m.group(2)
#     return p, None

# class UF:
#     def __init__(s): s.p = {}
#     def f(s, x):
#         s.p.setdefault(x, x)
#         while s.p[x] != x: s.p[x] = s.p[s.p[x]]; x = s.p[x]
#         return x
#     def u(s, a, b): s.p[s.f(a)] = s.f(b)

# parsed = [{c: [(p, *parse(p)) for p in split_cell(r[c])] for c in COLS} for _, r in df.iterrows()]
# ufs = {}
# for c in COLS:
#     uf = UF(); keys = set()
#     for row in parsed:
#         for raw, base, acr in row[c]:
#             kb = norm(base); keys.add(kb); uf.f(kb)
#             if acr: ka = norm(acr); keys.add(ka); uf.u(kb, ka)
#     for k in list(keys):
#         w = k.split()
#         if len(w) >= 2:
#             for ini in {"".join(x[0] for x in w if x not in STOP), "".join(x[0] for x in w)}:
#                 if len(ini) >= 2 and ini in keys and ini != k: uf.u(k, ini)
#     if FUZZY < 1:
#         ks = sorted(k for k in keys if len(k) >= 5 and not re.search(r"\d", k))
#         for i, a in enumerate(ks):
#             for b in ks[i+1:]:
#                 if abs(len(a) - len(b)) > 3: continue
#                 r = difflib.SequenceMatcher(None, a, b).ratio()
#                 if r >= FUZZY and uf.f(a) != uf.f(b):
#                     uf.u(a, b); print("  fuzzy merge [%s]: '%s' ~ '%s' (%.2f)" % (c, a, b, r))
#     ufs[c] = uf

# nodes, variants, rows_ids = {}, defaultdict(Counter), []
# for row in parsed:
#     ids = []
#     for c in COLS:
#         for raw, base, acr in row[c]:
#             key = ufs[c].f(norm(base))
#             nid = key if MERGE_ACROSS_COLS else c + "::" + key
#             nodes.setdefault(nid, {"id": nid, "col": c, "rows": 0})
#             variants[nid][raw] += 1
#             if nid not in ids: ids.append(nid)
#     for nid in ids: nodes[nid]["rows"] += 1
#     rows_ids.append(ids)
# for nid, n in nodes.items():
#     vs = variants[nid]
#     n["label"] = max(vs, key=lambda s: (len(s), vs[s]))
#     n["aliases"] = sorted(v for v in vs if v != n["label"])

# # ───────── GEOGRAPHY ─────────
# def fetch_first(urls):
#     for u in urls:
#         try: return json.loads(urllib.request.urlopen(u, timeout=40).read().decode("utf-8"))
#         except Exception as e: print("  !! could not download", u, "->", e)
#     return None
# states_gj, world_gj = fetch_first(STATES_URLS), fetch_first(WORLD_URLS)

# def _ring(r):
#     a = cx = cy = 0.0
#     for (x0, y0, *_), (x1, y1, *_) in zip(r, r[1:] + r[:1]):
#         c = x0 * y1 - x1 * y0; a += c; cx += (x0 + x1) * c; cy += (y0 + y1) * c
#     a /= 2
#     return (abs(a), (cx / (6 * a), cy / (6 * a))) if a else (0, (r[0][0], r[0][1]))
# def centroid(g):
#     polys = [g["coordinates"]] if g["type"] == "Polygon" else g["coordinates"]
#     lon, lat = max((_ring(p[0]) for p in polys), key=lambda t: t[0])[1]
#     return lat, lon
# def rnd(c): return [round(c[0], 2), round(c[1], 2)] if isinstance(c[0], (int, float)) else [rnd(x) for x in c]

# FEAT, cent = {}, {}
# for pref, gj in (("S:", states_gj), ("C:", world_gj)):
#     for f in (gj or {}).get("features", []):
#         nm = f["properties"].get("name")
#         if not nm or not f.get("geometry"): continue
#         fid = pref + nm
#         FEAT[fid] = {"type": "Feature", "properties": {"fid": fid, "name": nm},
#                      "geometry": {"type": f["geometry"]["type"], "coordinates": rnd(f["geometry"]["coordinates"])}}
#         cent[fid] = centroid(f["geometry"])

# LK = {}
# for fid in cent:
#     if fid.startswith("C:"): LK[norm(fid[2:])] = ("country", [fid])
# for k, vs in COUNTRY_ALIAS.items():
#     for v in vs:
#         if "C:" + v in cent: LK[norm(k)] = ("country", ["C:" + v]); break
# for fid in cent:
#     if fid.startswith("S:"): LK[norm(fid[2:])] = ("state", [fid])
# for k, v in REGIONS.items():
#     mem = ["S:" + n for n in v if "S:" + n in cent]
#     if mem: LK[norm(k)] = ("region", mem)

# def mk(kind, mem):
#     return {"kind": kind, "members": mem, "lat": sum(cent[m][0] for m in mem) / len(mem), "lon": sum(cent[m][1] for m in mem) / len(mem)}
# def resolve_one(s):
#     k = norm(s)
#     if not k: return None, ""
#     if k in GLOBAL: return {"kind": "global", "members": ["*"], "lat": 0.0, "lon": -30.0}, "exact"
#     if k in LK: return mk(*LK[k]), "exact"
#     st = s.strip()
#     if st in ab2name and "S:" + ab2name[st] in cent: return mk("state", ["S:" + ab2name[st]]), "abbr"
#     for key in sorted(LK, key=len, reverse=True):
#         if len(key) >= 4 and re.search(r"\b" + re.escape(key) + r"\b", k): return mk(*LK[key]), "contains '" + key + "'"
#     c = difflib.get_close_matches(k, list(LK), n=1, cutoff=GEO_FUZZY)
#     if c: return mk(*LK[c[0]]), "fuzzy '" + c[0] + "'"
#     return None, ""

# unresolved, soft, ngeo = [], [], 0
# for nid, n in nodes.items():
#     if n["col"] not in GEO_COLS: continue
#     for s in [n["label"]] + n["aliases"]:
#         g, how = resolve_one(s)
#         if g:
#             n["geo"] = g; ngeo += 1
#             if how != "exact": soft.append("%s -> %s [%s]" % (s, ",".join(m[2:] for m in g["members"][:4]), how))
#             break
#     else: unresolved.append(n["label"])
# print("geo nodes resolved: %d | unresolved: %d" % (ngeo, len(unresolved)))
# if soft: print("non-exact matches (verify):\n  " + "\n  ".join(soft))
# if unresolved: print("UNRESOLVED (add to REGIONS / ALIASES):", unresolved)

# rows = []
# for ids in rows_ids:
#     geo = sorted({m for nid in ids if "geo" in nodes[nid] for m in nodes[nid]["geo"]["members"]})
#     rows.append({"ids": ids, "geo": geo})
# print("rows with geography: %d/%d" % (sum(bool(r["geo"]) for r in rows), len(rows)))

# used_c = {m for n in nodes.values() if "geo" in n for m in n["geo"]["members"] if m.startswith("C:")}
# fc = lambda fids: {"type": "FeatureCollection", "features": [FEAT[f] for f in fids if f in FEAT]}
# colors = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(COLS)}
# cnt = Counter(n["col"] for n in nodes.values())
# data = {"cols": COLS, "colors": colors, "counts": {c: cnt[c] for c in COLS}, "hub": HUB_COL,
#         "nodes": list(nodes.values()), "rows": rows,
#         "geo": {"states": fc([f for f in FEAT if f.startswith("S:")]), "countries": fc(sorted(used_c))},
#         "gibs": {"date": IMAGERY_DATE, "max": date.today().isoformat(), "base": GIBS_BASE, "over": GIBS_OVER}}

# # ───────── HTML ─────────
# HTML = r"""<!doctype html><html><head><meta charset="utf-8"><title>Network v2.1</title>
# <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
# <style>
# html,body{margin:0;height:100%;background:#0b0e14;color:#dde3ee;font:12px/1.35 system-ui,sans-serif;overflow:hidden}
# #graph,#map{position:fixed;inset:0}#map{display:none;z-index:0;background:#0b0e14}
# .p{position:fixed;top:8px;background:#141923ee;border:1px solid #2a3142;border-radius:8px;padding:10px;overflow-y:auto;z-index:1000}
# #panel{left:8px;width:262px;max-height:calc(100vh - 36px)}
# #info{right:8px;width:290px;display:none;max-height:calc(100vh - 120px)}
# h4{margin:10px 0 4px;font-size:11px;text-transform:uppercase;letter-spacing:.06em;color:#8a93a6}h4:first-child{margin-top:0}
# label.col{display:flex;align-items:center;gap:6px;padding:1px 0;cursor:pointer}
# label.col i{width:10px;height:10px;border-radius:50%;display:inline-block}label.col span{margin-left:auto;color:#6b7488}
# input[type=range]{width:100%;margin:2px 0}select,input[type=text],input[type=date],button{background:#1d2431;color:#dde3ee;border:1px solid #333c50;border-radius:5px;padding:4px 6px;font-size:12px;color-scheme:dark}
# select,input[type=text]{width:100%;box-sizing:border-box}button{cursor:pointer}button:hover{background:#2a3446}button.on{background:#2ec4b6;color:#000;border-color:#2ec4b6}
# .row{display:flex;gap:4px;margin:3px 0;align-items:center}.row>*{flex:1}.v{float:right;color:#8a93a6}
# #info a{color:#7cc4ff;cursor:pointer;display:block;padding:1px 0}#info a:hover{text-decoration:underline}
# #stats{position:fixed;top:8px;left:50%;transform:translateX(-50%);background:#141923cc;padding:4px 10px;border-radius:12px;color:#8a93a6;z-index:1000;white-space:nowrap}
# .chip{display:inline-block;background:#2a3446;border:1px solid #ffd166;border-radius:10px;padding:1px 7px;margin:2px 2px 0 0;cursor:pointer}
# .leaflet-container{font:inherit}.lbl{background:#000a;color:#fff;border:0;box-shadow:none;padding:1px 4px;font-size:11px}
# .lbl:before{display:none}
# </style></head><body>
# <div id="graph"></div><div id="map"></div>
# <div id="panel" class="p">
#  <div class="row"><button id="vNet" class="on">3D network</button><button id="vMap">Map</button></div>
#  <label class="col"><input type="checkbox" id="pin"> pin geography to map (3D view)</label>
#  <h4>Search</h4><input type="text" id="q" list="dl" placeholder="node / alias, Enter"><datalist id="dl"></datalist>
#  <h4>Geography filter <span class="v"><a href="#" id="gclr">clear</a></span></h4>
#  <select id="gadd"><option value="">+ add place…</option></select><div id="chips"></div>
#  <div style="color:#6b7488">or click states / countries on the Map</div>
#  <h4>Map imagery (NASA GIBS)</h4>
#  <div class="row"><button id="dprev" style="flex:1">◀</button><input type="date" id="gdate" style="flex:4"><button id="dnext" style="flex:1">▶</button></div>
#  <div>Data-overlay opacity</div><input type="range" id="iop" min="0.1" max="1" step="0.05" value="0.7">
#  <label class="col"><input type="checkbox" id="shade" checked> shade geographies by activity</label>
#  <div>Network edges (map)</div><select id="edges"><option value="auto">auto</option><option value="all">all</option><option value="sel">selected node only</option><option value="none">none</option></select>
#  <div style="color:#6b7488">basemap / overlay picker: bottom-right of map</div>
#  <h4>Columns <span class="v"><a href="#" id="all">all</a> · <a href="#" id="none">none</a></span></h4><div id="legend"></div>
#  <h4>Link mode</h4>
#  <select id="mode"><option value="all">All pairs in a row</option><option value="hub">Hub column → everything</option><option value="chain">Chain by column order</option></select>
#  <div class="row"><select id="hub"></select></div>
#  <label class="col"><input type="checkbox" id="intra" checked> link values within same column</label>
#  <h4>Filters</h4>
#  <div>Min edge weight <span class="v" id="vW">1</span></div><input type="range" id="minW" min="1" max="20" value="1">
#  <div>Min node degree <span class="v" id="vD">0</span></div><input type="range" id="minDeg" min="0" max="30" value="0">
#  <h4>Appearance</h4>
#  <label class="col"><input type="checkbox" id="labels"> text labels</label>
#  <div>Label min degree <span class="v" id="vL">0</span></div><input type="range" id="labelDeg" min="0" max="30" value="0">
#  <div>Node size</div><input type="range" id="size" min="0.3" max="4" step="0.1" value="1">
#  <div>Link opacity (3D)</div><input type="range" id="op" min="0.05" max="1" step="0.05" value="0.35">
#  <label class="col"><input type="checkbox" id="part"> flow particles (3D)</label>
#  <label class="col"><input type="checkbox" id="rot"> auto-rotate (3D)</label>
#  <h4>Physics (3D)</h4>
#  <div>Link distance</div><input type="range" id="dist" min="10" max="300" value="40">
#  <div>Repulsion</div><input type="range" id="chg" min="5" max="400" value="30">
#  <div class="row"><button id="dim">2D / 3D</button><button id="fit">Fit view</button><button id="snap">PNG</button></div>
#  <div style="color:#6b7488;margin-top:6px">Esc = clear selection</div>
# </div>
# <div id="info" class="p"></div><div id="stats"></div>
# <script src="https://unpkg.com/three@0.150.1/build/three.min.js"></script>
# <script src="https://unpkg.com/three-spritetext@1.8.0/dist/three-spritetext.min.js"></script>
# <script src="https://unpkg.com/3d-force-graph@1.73.2/dist/3d-force-graph.min.js"></script>
# <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
# <script>
# const D=__DATA__, $=id=>document.getElementById(id);
# const esc=s=>String(s).replace(/[&<>"]/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));
# const NODE=new Map(D.nodes.map(n=>[n.id,n])), cache=new Map(), CUR={links:[],nodes:[],act:[],anch:new Map(),ver:0,lay:null,stat:''};
# const S={vis:new Set(D.cols),mode:'all',hub:D.hub,minW:1,minDeg:0,intra:true,labels:false,labelDeg:0,size:1,hl:null,sel:null,dim:3,
#  view:'net',pin:false,geoSel:new Set(),shade:true,edges:'auto',date:D.gibs.date,iop:.7};
# const sid=x=>typeof x==='object'?x.id:x;
# const K=Math.cos(38*Math.PI/180), SC=2.2, proj=(lat,lon)=>[lon*K*SC,lat*SC];
# const hsh=s=>{let h=2166136261;for(let i=0;i<s.length;i++){h^=s.charCodeAt(i);h=Math.imul(h,16777619)}return(h>>>0)/4294967295};
# const jit=(id,k)=>[(hsh(id+'a')-.5)*k,(hsh(id+'b')-.5)*k];

# // ───────── geography row filter + anchors ─────────
# function passG(r){if(!S.geoSel.size)return true;if(r.geo.includes('*'))return true;return r.geo.some(g=>S.geoSel.has(g))}
# function anchors(rows){   // geo nodes -> centroid; other nodes -> mean of the geographies in their rows
#  const A=new Map(),sum=new Map();
#  for(const n of D.nodes)if(n.geo&&n.geo.kind!=='global')A.set(n.id,[n.geo.lat,n.geo.lon]);
#  for(const r of rows){
#   const g=r.ids.filter(i=>A.has(i));if(!g.length)continue;
#   const m=[g.reduce((s,i)=>s+A.get(i)[0],0)/g.length,g.reduce((s,i)=>s+A.get(i)[1],0)/g.length];
#   for(const id of r.ids){if(NODE.get(id).geo)continue;const s=sum.get(id)||[0,0,0];s[0]+=m[0];s[1]+=m[1];s[2]++;sum.set(id,s)}}
#  for(const [id,s] of sum)A.set(id,[s[0]/s[2],s[1]/s[2]]);
#  return A}

# // ───────── 3D graph ─────────
# function labelObj(n){
#  if(!S.labels||n.deg<S.labelDeg||typeof SpriteText==='undefined')return null;
#  const s=new SpriteText(n.label,3.2,D.colors[n.col]);
#  s.position.y=Math.cbrt(Math.max(1,Math.sqrt(n.deg+1))*S.size)*4+4;s.material.depthWrite=false;return s}
# function nodeTip(n){return `<div style="background:#000d;padding:4px 8px;border-radius:4px;font:12px system-ui"><b>${esc(n.label)}</b><br>${esc(n.col)} · ${n.deg} links · ${n.rows} rows${n.aliases.length?'<br><i>aka '+esc(n.aliases.join(', '))+'</i>':''}</div>`}
# const G=ForceGraph3D({controlType:'trackball'})($('graph')).backgroundColor('#0b0e14').nodeRelSize(4)
#  .nodeVal(n=>Math.max(1,Math.sqrt(n.deg+1))*S.size)
#  .nodeColor(n=>S.hl&&!S.hl.n.has(n.id)?'#262b36':D.colors[n.col])
#  .nodeLabel(nodeTip)
#  .linkColor(l=>S.hl&&!S.hl.l.has(l)?'#1b1f29':'#8a93a6').linkOpacity(.35)
#  .linkWidth(l=>S.hl&&S.hl.l.has(l)?2:Math.min(.3+l.w*.35,4)).linkDirectionalParticleWidth(1.6)
#  .nodeThreeObjectExtend(true).nodeThreeObject(labelObj).onNodeClick(pick).onBackgroundClick(()=>select(null));
# const anchorForce=()=>{let ns=[];const f=a=>{if(!S.pin)return;for(const n of ns){if(n.ax==null||n.geo)continue;
#  n.vx+=(n.ax-n.x)*.3*a;n.vy+=(n.ay-n.y)*.3*a;if(S.dim===3)n.vz+=(0-n.z)*.03*a}};f.initialize=x=>{ns=x};return f};
# G.d3Force('anchor',anchorForce());
# let fitted=false;G.onEngineStop(()=>{if(!fitted){fitted=true;G.zoomToFit(600,60)}});
# const refresh=()=>{if(S.view==='map'){renderMap();return}G.nodeColor(G.nodeColor());G.linkColor(G.linkColor());G.linkWidth(G.linkWidth())};

# let OUT=null;
# function outlines(on){
#  const sc=G.scene();if(OUT){sc.remove(OUT);OUT.geometry.dispose();OUT=null}if(!on)return;
#  const pts=[],ring=r=>{for(let i=0;i<r.length-1;i++){if(Math.abs(r[i][0]-r[i+1][0])>180)continue;
#   const a=proj(r[i][1],r[i][0]),b=proj(r[i+1][1],r[i+1][0]);pts.push(a[0],a[1],0,b[0],b[1],0)}};
#  for(const fc of [D.geo.states,D.geo.countries])for(const f of fc.features){const g=f.geometry;(g.type==='Polygon'?[g.coordinates]:g.coordinates).forEach(p=>p.forEach(ring))}
#  const geo=new THREE.BufferGeometry();geo.setAttribute('position',new THREE.Float32BufferAttribute(pts,3));
#  OUT=new THREE.LineSegments(geo,new THREE.LineBasicMaterial({color:0x3a4a6b,transparent:true,opacity:.75}));sc.add(OUT)}
# function applyPin(){
#  for(const n of CUR.nodes){const a=CUR.anch.get(n.id);
#   if(S.pin&&a){const p=proj(a[0],a[1]);n.ax=p[0];n.ay=p[1];
#    if(n.geo){const j=jit(n.id,5);n.fx=p[0]+j[0];n.fy=p[1]+j[1];n.fz=0}
#    else{n.fx=n.fy=n.fz=null;if(n.x==null){n.x=p[0]+Math.random()*4;n.y=p[1]+Math.random()*4;n.z=Math.random()*10}}}
#   else{n.fx=n.fy=n.fz=null;n.ax=n.ay=null}}}

# // ───────── build ─────────
# function build(){
#  const rows=CUR.act=D.rows.filter(passG);
#  const W=new Map(),cnt=new Map(),add=(a,b)=>{if(a===b)return;const k=a<b?a+'\u0001'+b:b+'\u0001'+a;W.set(k,(W.get(k)||0)+1)};
#  const clique=L=>{for(let i=0;i<L.length;i++)for(let j=i+1;j<L.length;j++)add(L[i],L[j])};
#  for(const row of rows){
#   const ids=row.ids.filter(i=>S.vis.has(NODE.get(i).col)),col=i=>NODE.get(i).col;
#   ids.forEach(i=>cnt.set(i,(cnt.get(i)||0)+1));
#   if(S.mode==='all'){for(let i=0;i<ids.length;i++)for(let j=i+1;j<ids.length;j++){if(!S.intra&&col(ids[i])===col(ids[j]))continue;add(ids[i],ids[j])}}
#   else if(S.mode==='hub'){const h=ids.filter(i=>col(i)===S.hub),r=ids.filter(i=>col(i)!==S.hub);h.forEach(a=>r.forEach(b=>add(a,b)))}
#   else{const by={};ids.forEach(i=>(by[col(i)]??=[]).push(i));const cs=D.cols.filter(c=>by[c]);
#    for(let k=0;k<cs.length-1;k++)by[cs[k]].forEach(a=>by[cs[k+1]].forEach(b=>add(a,b)));if(S.intra)cs.forEach(c=>clique(by[c]))}}
#  const raw=[],deg=new Map();
#  for(const [k,w] of W){if(w<S.minW)continue;const [a,b]=k.split('\u0001');raw.push({source:a,target:b,w});deg.set(a,(deg.get(a)||0)+1);deg.set(b,(deg.get(b)||0)+1)}
#  const nodes=D.nodes.filter(n=>S.vis.has(n.col)&&cnt.has(n.id)&&(deg.get(n.id)||0)>=S.minDeg).map(n=>{
#   let o=cache.get(n.id);if(!o){o={...n};cache.set(n.id,o)}o.deg=deg.get(n.id)||0;o.rows=cnt.get(n.id);return o});
#  const keep=new Set(nodes.map(n=>n.id)),links=raw.filter(l=>keep.has(l.source)&&keep.has(l.target));
#  CUR.links=links;CUR.nodes=nodes;CUR.ver++;CUR.anch=anchors(rows);applyPin();
#  G.graphData({nodes,links});G.nodeThreeObject(labelObj);
#  CUR.stat=`${nodes.length} nodes · ${links.length} edges · ${rows.length}/${D.rows.length} rows`;$('stats').textContent=CUR.stat;
#  const s=S.sel&&cache.get(S.sel);if(s&&keep.has(s.id))select(s);else select(null)}
# function select(n){
#  S.sel=n?n.id:null;
#  if(!n){S.hl=null;$('info').style.display='none';refresh();return}
#  const ns=new Set([n.id]),ls=new Set(),by={};
#  for(const l of CUR.links){const a=sid(l.source),b=sid(l.target);
#   if(a===n.id||b===n.id){ls.add(l);const o=a===n.id?b:a;ns.add(o);(by[NODE.get(o).col]??=[]).push([o,l.w])}}
#  S.hl={n:ns,l:ls};
#  let h=`<h4>${esc(n.col)}</h4><b style="font-size:14px;color:${D.colors[n.col]}">${esc(n.label)}</b>`;
#  if(n.aliases.length)h+=`<div><i>aka ${esc(n.aliases.join(', '))}</i></div>`;
#  h+=`<div>${n.rows} rows · ${ns.size-1} connections</div>`;
#  if(n.geo)h+=`<div>geo: ${n.geo.kind==='global'?'global':esc(n.geo.members.length>6?n.geo.members.length+' areas':n.geo.members.map(m=>m.slice(2)).join(', '))}</div>`;
#  for(const c of D.cols){const L=by[c];if(!L)continue;L.sort((a,b)=>b[1]-a[1]);
#   h+=`<h4><span style="color:${D.colors[c]}">●</span> ${esc(c)} (${L.length})</h4>`+L.map(([o,w])=>`<a data-id="${esc(o)}">${esc(NODE.get(o).label)} <span style="color:#6b7488">×${w}</span></a>`).join('')}
#  $('info').innerHTML=h;$('info').style.display='block';refresh()}
# function pick(n){select(n);if(!n)return;
#  if(S.view==='map'){const p=CUR.lay&&CUR.lay.P.get(n.id);if(p)M.flyTo(p,Math.max(M.getZoom(),5));return}
#  const x=n.x||0,y=n.y||0,z=n.z||0,d=Math.hypot(x,y,z)||1,r=1+120/d;G.cameraPosition({x:x*r,y:y*r,z:z*r},{x,y,z},800)}
# $('info').onclick=e=>{const o=cache.get(e.target.dataset.id);if(o)pick(o)};
# addEventListener('keydown',e=>{if(e.key==='Escape')select(null)});

# // ───────── Map view: Leaflet + NASA GIBS ─────────
# let M=null,GL=null,NL=null,LC=null;const GF={},TL=[],OVR=[];
# function gibsUrl(g){const t=g[4]==='date'?S.date:g[4];
#  return `https://gibs.earthdata.nasa.gov/wmts/epsg3857/best/${g[1]}/default/${t?t+'/':''}${g[2]}/{z}/{y}/{x}.${g[3]}`}
# function gibsLayer(g,over){
#  const mz=+g[2].match(/Level(\d+)/)[1];
#  const l=L.tileLayer(gibsUrl(g),{maxNativeZoom:mz,maxZoom:14,tileSize:256,noWrap:true,bounds:[[-85.05,-180],[85.05,180]],
#   attribution:'Imagery: NASA GIBS / Worldview',zIndex:over?5:1,opacity:over&&g[5]?S.iop:1});
#  l._g=g;TL.push(l);if(over&&g[5])OVR.push(l);return l}
# function initMap(){
#  if(M)return;
#  M=L.map('map',{preferCanvas:true,minZoom:2,maxZoom:13,zoomSnap:.5}).setView([39,-96],4);
#  const base={},over={};
#  D.gibs.base.forEach((g,i)=>{const l=gibsLayer(g,false);base[g[0]]=l;if(i===0)l.addTo(M)});
#  base['No imagery (dark)']=L.layerGroup();
#  D.gibs.over.forEach(g=>{const l=gibsLayer(g,true);over[g[0]]=l;if(g[6])l.addTo(M)});
#  LC=L.control.layers(base,over,{position:'bottomright',collapsed:true}).addTo(M);
#  GL=L.layerGroup().addTo(M);
#  for(const fc of [D.geo.states,D.geo.countries])
#   L.geoJSON(fc,{style:()=>({weight:.7,color:'#cfd8e3',opacity:.6,fillOpacity:0}),onEachFeature:(f,l)=>{GF[f.properties.fid]=l;
#    l.on('click',()=>toggleGeo(f.properties.fid));l.bindTooltip('',{sticky:true})}}).addTo(GL);
#  NL=L.layerGroup().addTo(M);
#  M.on('zoomend',()=>{if(S.view==='map')renderMap()})}
# function updateChoro(){
#  const c={};let mx=1;
#  for(const r of CUR.act)for(const g of r.geo){if(g==='*')continue;c[g]=(c[g]||0)+1;if(c[g]>mx)mx=c[g]}
#  const foot=new Set();
#  if(S.sel){const n=NODE.get(S.sel);if(n&&n.geo)n.geo.members.forEach(m=>foot.add(m));
#   for(const r of CUR.act)if(r.ids.includes(S.sel))r.geo.forEach(g=>foot.add(g))}
#  for(const [fid,l] of Object.entries(GF)){const v=c[fid]||0,sel=S.geoSel.has(fid),f=foot.has(fid);
#   l.setStyle({fillColor:f?'#ffd166':'#2ec4b6',fillOpacity:f?.38:(S.shade&&v?.08+.42*v/mx:0),weight:sel?3:f?1.6:.7,
#    color:(sel||f)?'#ffd166':'#cfd8e3',opacity:(sel||f)?1:.6});
#   l.setTooltipContent(`<b>${esc(l.feature.properties.name)}</b><br>${v} rows`+(sel?'<br>(geo filter on)':'<br>click to filter'))}}
# function layout(){   // sunflower packing in pixel space around each anchor: no overlaps, stable across zoom
#  const key=CUR.ver+'|'+M.getZoom();if(CUR.lay&&CUR.lay.key===key)return CUR.lay;
#  const grp=new Map();
#  for(const n of CUR.nodes){const a=CUR.anch.get(n.id);if(!a)continue;const k=Math.round(a[0]*2)+','+Math.round(a[1]*2);
#   if(!grp.has(k))grp.set(k,[]);grp.get(k).push([n,a])}
#  const P=new Map(),pts=[];
#  for(const g of grp.values()){
#   g.sort((x,y)=>x[0].col.localeCompare(y[0].col)||x[0].label.localeCompare(y[0].label));
#   const la=g.reduce((s,t)=>s+t[1][0],0)/g.length,lo=g.reduce((s,t)=>s+t[1][1],0)/g.length,c=M.latLngToLayerPoint([la,lo]);
#   g.forEach(([n],i)=>{const r=g.length===1?0:11*Math.sqrt(i+.5),a=i*2.399963;
#    const ll=M.layerPointToLatLng(L.point(c.x+r*Math.cos(a),c.y+r*Math.sin(a)));P.set(n.id,[ll.lat,ll.lng]);pts.push(n)})}
#  return CUR.lay={key,P,pts}}
# function renderMap(){
#  initMap();NL.clearLayers();updateChoro();
#  const {P,pts}=layout(),em=S.edges==='auto'?(CUR.links.length>2500?'sel':'all'):S.edges;
#  if(em!=='none')for(const l of CUR.links){const a=P.get(sid(l.source)),b=P.get(sid(l.target));if(!a||!b)continue;
#   const on=S.hl&&S.hl.l.has(l);if(em==='sel'&&!on)continue;
#   L.polyline([a,b],{weight:on?2.4:Math.min(.5+l.w*.3,3),color:on?'#ffd166':'#ffffff',opacity:on?.95:(S.hl?.07:.28),interactive:false}).addTo(NL)}
#  let nl=0;
#  for(const n of pts){const dim=S.hl&&!S.hl.n.has(n.id);
#   const m=L.circleMarker(P.get(n.id),{radius:4+Math.sqrt(n.deg+1)*S.size*1.3,color:'#fff',weight:1,opacity:dim?.25:.9,fillColor:dim?'#555':D.colors[n.col],fillOpacity:dim?.35:.95});
#   if(S.labels&&n.deg>=S.labelDeg&&!dim&&nl<250){nl++;m.bindTooltip(esc(n.label),{permanent:true,direction:'right',offset:[6,0],className:'lbl'})}
#   else m.bindTooltip(nodeTip(n));
#   m.on('click',e=>{L.DomEvent.stopPropagation(e);select(n)});m.addTo(NL)}
#  const miss=CUR.nodes.length-pts.length;
#  $('stats').textContent=CUR.stat+(miss?` · ${miss} nodes without geography hidden`:'')}
# function fitMap(){const {P}=layout(),v=[...P.values()];if(v.length)M.fitBounds(L.latLngBounds(v),{padding:[60,60],maxZoom:7})}
# function setView(v){
#  S.view=v;$('graph').style.display=v==='net'?'block':'none';$('map').style.display=v==='map'?'block':'none';
#  $('vNet').classList.toggle('on',v==='net');$('vMap').classList.toggle('on',v==='map');
#  if(v==='map'){G.pauseAnimation();initMap();M.invalidateSize();renderMap();if(!M._fitted&&CUR.lay&&CUR.lay.pts.length){M._fitted=1;fitMap()}}
#  else{G.resumeAnimation();G.width(innerWidth).height(innerHeight);$('stats').textContent=CUR.stat}}

# // ───────── imagery controls ─────────
# const setDate=d=>{S.date=d;$('gdate').value=d;TL.filter(l=>l._g[4]==='date').forEach(l=>l.setUrl(gibsUrl(l._g)))};
# $('gdate').value=D.gibs.date;$('gdate').max=D.gibs.max;
# $('gdate').onchange=e=>{if(e.target.value)setDate(e.target.value)};
# const shiftDay=k=>{const d=new Date(S.date+'T00:00:00Z');d.setUTCDate(d.getUTCDate()+k);const s=d.toISOString().slice(0,10);if(s<=D.gibs.max)setDate(s)};
# $('dprev').onclick=()=>shiftDay(-1);$('dnext').onclick=()=>shiftDay(1);
# $('iop').oninput=e=>{S.iop=+e.target.value;OVR.forEach(l=>l.setOpacity(S.iop))};
# $('shade').onchange=e=>{S.shade=e.target.checked;if(S.view==='map')renderMap()};
# $('edges').onchange=e=>{S.edges=e.target.value;if(S.view==='map')renderMap()};

# // ───────── geography filter UI ─────────
# const gname=fid=>(D.geo.states.features.concat(D.geo.countries.features).find(f=>f.properties.fid===fid)||{properties:{name:fid}}).properties.name;
# function renderChips(){$('chips').innerHTML=[...S.geoSel].map(f=>`<span class="chip" data-f="${esc(f)}">${esc(gname(f))} ×</span>`).join('')}
# function toggleGeo(f){S.geoSel.has(f)?S.geoSel.delete(f):S.geoSel.add(f);renderChips();build()}
# for(const fc of [D.geo.states,D.geo.countries])
#  [...fc.features].sort((a,b)=>a.properties.name.localeCompare(b.properties.name)).forEach(f=>{const o=document.createElement('option');
#   o.value=f.properties.fid;o.textContent=f.properties.name+(f.properties.fid.startsWith('C:')?' (country)':'');$('gadd').append(o)});
# $('gadd').onchange=e=>{if(e.target.value&&!S.geoSel.has(e.target.value))toggleGeo(e.target.value);e.target.value=''};
# $('chips').onclick=e=>{if(e.target.dataset.f)toggleGeo(e.target.dataset.f)};
# $('gclr').onclick=e=>{e.preventDefault();S.geoSel.clear();renderChips();build()};

# // ───────── general UI ─────────
# D.cols.forEach(c=>{const l=document.createElement('label');l.className='col';
#  l.innerHTML=`<input type="checkbox" checked data-c="${esc(c)}"><i style="background:${D.colors[c]}"></i>${esc(c)}<span>${D.counts[c]}</span>`;$('legend').append(l);
#  const o=document.createElement('option');o.value=o.textContent=c;if(c===D.hub)o.selected=true;$('hub').append(o)});
# D.nodes.forEach(n=>{const o=document.createElement('option');o.value=n.label;$('dl').append(o)});
# $('legend').onchange=e=>{const c=e.target.dataset.c;e.target.checked?S.vis.add(c):S.vis.delete(c);build()};
# const setAll=v=>{document.querySelectorAll('#legend input').forEach(i=>{i.checked=v;v?S.vis.add(i.dataset.c):S.vis.delete(i.dataset.c)});build()};
# $('all').onclick=e=>{e.preventDefault();setAll(true)};$('none').onclick=e=>{e.preventDefault();setAll(false)};
# $('mode').onchange=e=>{S.mode=e.target.value;build()};$('hub').onchange=e=>{S.hub=e.target.value;build()};
# $('intra').onchange=e=>{S.intra=e.target.checked;build()};
# $('minW').oninput=e=>{S.minW=+e.target.value;$('vW').textContent=S.minW;build()};
# $('minDeg').oninput=e=>{S.minDeg=+e.target.value;$('vD').textContent=S.minDeg;build()};
# $('labels').onchange=e=>{S.labels=e.target.checked;G.nodeThreeObject(labelObj);if(S.view==='map')renderMap()};
# $('labelDeg').oninput=e=>{S.labelDeg=+e.target.value;$('vL').textContent=S.labelDeg;G.nodeThreeObject(labelObj);if(S.view==='map')renderMap()};
# $('size').oninput=e=>{S.size=+e.target.value;G.nodeVal(G.nodeVal());G.nodeThreeObject(labelObj);if(S.view==='map')renderMap()};
# $('op').oninput=e=>G.linkOpacity(+e.target.value);
# $('part').onchange=e=>G.linkDirectionalParticles(e.target.checked?2:0);
# $('dist').oninput=e=>{G.d3Force('link').distance(+e.target.value);G.d3ReheatSimulation()};
# $('chg').oninput=e=>{G.d3Force('charge').strength(-+e.target.value);G.d3ReheatSimulation()};
# let rot=null;$('rot').onchange=e=>{clearInterval(rot);if(e.target.checked)rot=setInterval(()=>{
#  const p=G.cameraPosition(),c=Math.cos(.004),s=Math.sin(.004);G.cameraPosition({x:p.x*c+p.z*s,y:p.y,z:-p.x*s+p.z*c})},30)};
# $('dim').onclick=()=>{S.dim=S.dim===3?2:3;G.numDimensions(S.dim);G.d3ReheatSimulation()};
# $('fit').onclick=()=>S.view==='map'?fitMap():G.zoomToFit(600,60);
# $('snap').onclick=()=>{if(S.view==='map'){alert('PNG export is 3D-only; screenshot the map.');return}
#  const r=G.renderer();r.render(G.scene(),G.camera());const a=document.createElement('a');a.href=r.domElement.toDataURL('image/png');a.download='network.png';a.click()};
# $('pin').onchange=e=>{S.pin=e.target.checked;outlines(S.pin);applyPin();G.d3ReheatSimulation();setTimeout(()=>G.zoomToFit(800,60),1500)};
# $('vNet').onclick=()=>setView('net');$('vMap').onclick=()=>setView('map');
# $('q').addEventListener('change',e=>{
#  const q=e.target.value.trim().toLowerCase();if(!q)return;
#  const n=D.nodes.find(x=>x.label.toLowerCase()===q)||D.nodes.find(x=>(x.label+' '+x.aliases.join(' ')).toLowerCase().includes(q));
#  if(!n){$('stats').textContent='no match';return}
#  if(!S.vis.has(n.col)){S.vis.add(n.col);document.querySelector(`#legend input[data-c="${CSS.escape(n.col)}"]`).checked=true;build()}
#  setTimeout(()=>{const o=cache.get(n.id);CUR.nodes.includes(o)?pick(o):$('stats').textContent='node hidden by filters'},300)});
# G.d3Force('link').distance(40);G.d3Force('charge').strength(-30);
# build();
# </script></body></html>"""

# html = HTML.replace("__DATA__", json.dumps(data, ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/"))
# open(OUT_HTML, "w", encoding="utf-8").write(html)
# print("✔ %s: %d nodes, %d rows, %.0f KB" % (OUT_HTML, len(nodes), len(rows), len(html) / 1024))
# try:
#     from google.colab import files; files.download(OUT_HTML)
# except Exception: pass

Saving Network_Viz_A9.csv to Network_Viz_A9 (1).csv
999 rows | sep = COMMA | geo cols: ['State', 'Area of Interest', 'Area of Interest_2']
  fuzzy merge [Area of Interest_2]: 'california' ~ 'califronia' (0.90)
  fuzzy merge [End users]: 'us forest service' ~ 'usda forest service' (0.94)
geo nodes resolved: 59 | unresolved: 0
non-exact matches (verify):
  Latvia and Sweden -> Latvia [contains 'latvia']
rows with geography: 15/999
✔ network_v2.html: 279 nodes, 999 rows, 160 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
# # ── Network Viz v2: network + geography (map / 3D-pinned) + timeline ──
# import pandas as pd, re, json, difflib, io, urllib.request
# from collections import Counter, defaultdict
# from datetime import date

# # ───────── CONFIG ─────────
# CSV_FILE   = None                       # None = upload dialog; or path / raw GitHub URL
# OUT_HTML   = "network_v2.html"
# HUB_COL    = "PI"
# GEO_COLS   = ["State", "Area of Interest", "Area of Interest_2"]
# TIME_COLS  = ["Temporal", "Temporal_2"]
# MERGE_ACROSS_COLS = False
# FUZZY      = 0.88                       # near-dup merge inside a column (1.0 = off)
# GEO_FUZZY  = 0.80                       # typo tolerance for place names ("Califronia")
# ALIASES    = {}                         # {"GOES-R": "GOES-ABI", ...}
# CURRENT_YEAR = date.today().year        # used for "2025–present"
# STATES_URL = "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json"
# WORLD_URL  = "https://raw.githubusercontent.com/johan/world.geo.json/master/countries.geo.json"
# NA = {"", "nan", "none", "n/a", "na", "-", "tbd", "null"}
# PALETTE = ["#e63946","#f4a261","#2a9d8f","#4361ee","#e9c46a","#9b5de5","#00bbf9","#f15bb5","#80ed99","#ff9f1c",
#            "#06d6a0","#c77dff","#ef476f","#118ab2","#ffd166","#8ac926","#ff595e","#6a994e","#bc6c25","#48cae4"]

# # ───────── GAZETTEER (edit REGIONS to taste; keys are case/punct-insensitive) ─────────
# ABBR = ("AL Alabama,AK Alaska,AZ Arizona,AR Arkansas,CA California,CO Colorado,CT Connecticut,DE Delaware,DC District of Columbia,"
#         "FL Florida,GA Georgia,HI Hawaii,ID Idaho,IL Illinois,IN Indiana,IA Iowa,KS Kansas,KY Kentucky,LA Louisiana,ME Maine,"
#         "MD Maryland,MA Massachusetts,MI Michigan,MN Minnesota,MS Mississippi,MO Missouri,MT Montana,NE Nebraska,NV Nevada,"
#         "NH New Hampshire,NJ New Jersey,NM New Mexico,NY New York,NC North Carolina,ND North Dakota,OH Ohio,OK Oklahoma,OR Oregon,"
#         "PA Pennsylvania,PR Puerto Rico,RI Rhode Island,SC South Carolina,SD South Dakota,TN Tennessee,TX Texas,UT Utah,VT Vermont,"
#         "VA Virginia,WA Washington,WV West Virginia,WI Wisconsin,WY Wyoming")
# ab2name = dict(x.split(" ", 1) for x in ABBR.split(","))
# R = lambda s: [ab2name[a] for a in s.split()]
# ALL48 = [n for a, n in ab2name.items() if a not in ("AK", "HI", "PR")]
# EAST_COAST = R("ME NH MA RI CT NY NJ DE MD VA NC SC GA FL")
# GULF = R("TX LA MS AL FL")
# MIDWEST = R("IL IN IA KS MI MN MO NE ND OH SD WI")
# REGIONS = {
#     "conus": ALL48, "contiguous us": ALL48, "contiguous united states": ALL48, "continental us": ALL48,
#     "continental united states": ALL48, "lower 48": ALL48,
#     "united states": ALL48 + ["Alaska", "Hawaii"], "us": ALL48 + ["Alaska", "Hawaii"], "usa": ALL48 + ["Alaska", "Hawaii"],
#     "conus coastal": R("ME NH MA RI CT NY NJ DE MD VA NC SC GA FL AL MS LA TX CA OR WA"),
#     "us east": R("ME NH VT MA RI CT NY NJ PA DE MD DC VA WV NC SC GA FL OH MI IN KY TN AL MS"),
#     "east coast": EAST_COAST, "us east coast": EAST_COAST, "eastern seaboard": EAST_COAST,
#     "west coast": R("CA OR WA"), "us west": R("WA OR CA NV ID UT AZ MT WY CO NM"),
#     "gulf coast": GULF, "gulf coasts": GULF, "gulf states": GULF,
#     "midwest": MIDWEST, "midwest united states": MIDWEST, "us midwest": MIDWEST, "midwestern us": MIDWEST,
#     "northeast": R("ME NH VT MA RI CT NY NJ PA"), "southeast": R("AL FL GA KY MS NC SC TN VA WV"),
#     "southwest": R("AZ NM TX OK"), "pacific northwest": R("WA OR ID"), "great lakes": R("MN WI IL IN MI OH PA NY"),
#     "great plains": R("ND SD NE KS OK"),
# }
# GLOBAL = {"global", "worldwide", "world"}
# COUNTRY_ALIAS = {"tanzania": "United Republic of Tanzania", "uk": "United Kingdom", "great britain": "United Kingdom",
#                  "russia": "Russia", "ivory coast": "Ivory Coast"}

# # ───────── LOAD ─────────
# if CSV_FILE is None:
#     from google.colab import files
#     up = files.upload(); CSV_FILE = next(iter(up)); raw_bytes = up[CSV_FILE]
# elif CSV_FILE.startswith("http"):
#     raw_bytes = urllib.request.urlopen(CSV_FILE, timeout=30).read()
# else:
#     raw_bytes = open(CSV_FILE, "rb").read()
# try:    text = raw_bytes.decode("utf-8-sig")
# except UnicodeDecodeError: text = raw_bytes.decode("latin-1")
# sep = "\t" if "\t" in text.split("\n", 1)[0] else ","
# df = pd.read_csv(io.StringIO(text), sep=sep, dtype=str, keep_default_na=False)
# df.columns = [c.strip() for c in df.columns]
# COLS = list(df.columns)
# lc = {c.lower(): c for c in COLS}
# GEO_COLS  = [lc[g.lower()] for g in GEO_COLS  if g.lower() in lc]
# TIME_COLS = [lc[g.lower()] for g in TIME_COLS if g.lower() in lc]
# HUB_COL = lc.get(HUB_COL.lower(), COLS[0])
# print(len(df), "rows | sep =", "TAB" if sep == "\t" else "COMMA", "| geo:", GEO_COLS, "| time:", TIME_COLS)

# # ───────── PARSE / NORMALIZE (same as v1) ─────────
# STOP = {"of", "and", "the", "for", "in", "on", "a", "an", "&"}
# norm = lambda s: re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()
# ALIAS_N = {norm(k): v for k, v in ALIASES.items()}

# def split_cell(v):
#     out = []
#     for p in re.split(r"[;\n|]|,(?![^()]*\))", str(v).strip()):
#         p = re.sub(r"^\s*(and|&)\s+", "", p.strip(), flags=re.I).strip(" ,.;:")
#         while p.endswith(")") and p.count(")") > p.count("("): p = p[:-1].strip()
#         while p.startswith("(") and p.count("(") > p.count(")"): p = p[1:].strip()
#         if p.lower() in NA: continue
#         out.append(ALIAS_N.get(norm(p), p))
#     return out

# def parse(p):
#     m = re.match(r"^(.*?)\s*\(([^()]+)\)\s*$", p)
#     if m and m.group(1) and re.fullmatch(r"[A-Za-z0-9&\-]{2,10}", m.group(2)) and sum(ch.isupper() for ch in m.group(2)) >= 2:
#         return m.group(1).strip(), m.group(2)
#     return p, None

# class UF:
#     def __init__(s): s.p = {}
#     def f(s, x):
#         s.p.setdefault(x, x)
#         while s.p[x] != x: s.p[x] = s.p[s.p[x]]; x = s.p[x]
#         return x
#     def u(s, a, b): s.p[s.f(a)] = s.f(b)

# # ───────── TEMPORAL PARSING ─────────
# UNIT = {"minute": 1/60, "min": 1/60, "hour": 1, "hr": 1, "day": 24, "week": 168, "month": 730, "year": 8760, "yr": 8760, "h": 1}
# UNIT_RE = r"(minutes?|mins?|hours?|hrs?|h|days?|weeks?|months?|years?|yrs?)"
# def _h(num, unit):
#     u = unit.lower()
#     for k in ("minute", "min", "hour", "hr", "day", "week", "month", "year", "yr", "h"):
#         if u.startswith(k): return float(num) * UNIT[k]
# KEYW = [(r"sub-?hourly", .5), (r"hourly", 1), (r"daily|per day", 24), (r"weekly", 168), (r"monthly", 730),
#         (r"seasonal", 2190), (r"annual|yearly", 8760), (r"decadal", 87600)]
# def parse_time(s):
#     """-> ('per',[y0,y1]) for year ranges | ('dur',[h_lo,h_hi]) for cadence/lead time in hours | None"""
#     t = s.lower().replace("–", "-").replace("—", "-").replace("−", "-")
#     ys = re.findall(r"(?<![\d.])(1[89]\d\d|20\d\d)(?![\d.])", t)
#     if ys and not re.search(r"\d\s*-?\s*" + UNIT_RE + r"\b", t):
#         yy = [int(y) for y in ys]
#         if len(yy) == 1 and re.search(r"present|ongoing|current|now|-\s*$", t): yy.append(CURRENT_YEAR)
#         return "per", [min(yy), max(yy)]
#     m = re.search(r"(\d+(?:\.\d+)?)\s*(?:-|to)\s*(\d+(?:\.\d+)?)\s*-?\s*" + UNIT_RE + r"\b", t)
#     if m:
#         a, b = _h(m[1], m[3]), _h(m[2], m[3]); return "dur", [min(a, b), max(a, b)]
#     m = re.search(r"(\d+(?:\.\d+)?)\s*-?\s*(hourly|daily|weekly|monthly|yearly|annual)", t)
#     if m:
#         v = float(m[1]) * {"hourly": 1, "daily": 24, "weekly": 168, "monthly": 730, "yearly": 8760, "annual": 8760}[m[2]]
#         return "dur", [v, v]
#     m = re.search(r"(\d+(?:\.\d+)?)\s*-?\s*" + UNIT_RE + r"\b", t)
#     if m:
#         v = _h(m[1], m[2]); return "dur", [v, v]
#     for pat, h in KEYW:
#         if re.search(pat, t): return "dur", [h, h]
#     return None

# def cad_class(lo, hi):
#     mid = (lo * hi) ** .5
#     return ("Sub-daily (<24 h)" if mid < 24 else "Daily-scale (1-7 d)" if mid < 168 else
#             "Weekly-monthly" if mid < 2190 else "Seasonal-annual+")

# parsed = [{c: [(p, *parse(p)) for p in split_cell(r[c])] for c in COLS} for _, r in df.iterrows()]
# if TIME_COLS:                                           # derived column: normalized cadence class
#     for row in parsed:
#         row["Cadence class"] = []
#         for c in TIME_COLS:
#             for raw, _, _ in row[c]:
#                 pt = parse_time(raw)
#                 if pt and pt[0] == "dur":
#                     k = cad_class(*pt[1])
#                     if all(k != x[0] for x in row["Cadence class"]): row["Cadence class"].append((k, k, None))
#     COLS.append("Cadence class")

# ufs = {}
# for c in COLS:
#     uf = UF(); keys = set()
#     for row in parsed:
#         for raw, base, acr in row[c]:
#             kb = norm(base); keys.add(kb); uf.f(kb)
#             if acr: ka = norm(acr); keys.add(ka); uf.u(kb, ka)
#     for k in list(keys):
#         w = k.split()
#         if len(w) >= 2:
#             for ini in {"".join(x[0] for x in w if x not in STOP), "".join(x[0] for x in w)}:
#                 if len(ini) >= 2 and ini in keys and ini != k: uf.u(k, ini)
#     if FUZZY < 1:
#         ks = sorted(k for k in keys if len(k) >= 5 and not re.search(r"\d", k))
#         for i, a in enumerate(ks):
#             for b in ks[i+1:]:
#                 if abs(len(a) - len(b)) > 3: continue
#                 r = difflib.SequenceMatcher(None, a, b).ratio()
#                 if r >= FUZZY and uf.f(a) != uf.f(b):
#                     uf.u(a, b); print("  fuzzy merge [%s]: '%s' ~ '%s' (%.2f)" % (c, a, b, r))
#     ufs[c] = uf

# nodes, variants, rows_ids = {}, defaultdict(Counter), []
# for row in parsed:
#     ids = []
#     for c in COLS:
#         for raw, base, acr in row[c]:
#             key = ufs[c].f(norm(base))
#             nid = key if MERGE_ACROSS_COLS else c + "::" + key
#             nodes.setdefault(nid, {"id": nid, "col": c, "rows": 0})
#             variants[nid][raw] += 1
#             if nid not in ids: ids.append(nid)
#     for nid in ids: nodes[nid]["rows"] += 1
#     rows_ids.append(ids)
# for nid, n in nodes.items():
#     vs = variants[nid]
#     n["label"] = max(vs, key=lambda s: (len(s), vs[s]))
#     n["aliases"] = sorted(v for v in vs if v != n["label"])

# # ───────── GEOGRAPHY ─────────
# def fetch(url):
#     try: return json.loads(urllib.request.urlopen(url, timeout=40).read().decode("utf-8"))
#     except Exception as e: print("  !! could not download", url, "->", e); return None
# states_gj, world_gj = fetch(STATES_URL), fetch(WORLD_URL)

# def _ring(r):
#     a = cx = cy = 0.0
#     for (x0, y0, *_), (x1, y1, *_) in zip(r, r[1:] + r[:1]):
#         c = x0 * y1 - x1 * y0; a += c; cx += (x0 + x1) * c; cy += (y0 + y1) * c
#     a /= 2
#     return (abs(a), (cx / (6 * a), cy / (6 * a))) if a else (0, (r[0][0], r[0][1]))
# def centroid(g):
#     polys = [g["coordinates"]] if g["type"] == "Polygon" else g["coordinates"]
#     lon, lat = max((_ring(p[0]) for p in polys), key=lambda t: t[0])[1]
#     return lat, lon
# def rnd(c): return [round(c[0], 2), round(c[1], 2)] if isinstance(c[0], (int, float)) else [rnd(x) for x in c]

# FEAT, cent = {}, {}
# for pref, gj in (("S:", states_gj), ("C:", world_gj)):
#     for f in (gj or {}).get("features", []):
#         nm = f["properties"].get("name")
#         if not nm or not f.get("geometry"): continue
#         fid = pref + nm
#         FEAT[fid] = {"type": "Feature", "properties": {"fid": fid, "name": nm},
#                      "geometry": {"type": f["geometry"]["type"], "coordinates": rnd(f["geometry"]["coordinates"])}}
#         cent[fid] = centroid(f["geometry"])

# LK = {}                                              # normalized name -> (kind, [feature ids]); regions > states > countries
# for fid in cent:
#     if fid.startswith("C:"): LK[norm(fid[2:])] = ("country", [fid])
# for k, v in COUNTRY_ALIAS.items():
#     if "C:" + v in cent: LK[norm(k)] = ("country", ["C:" + v])
# for fid in cent:
#     if fid.startswith("S:"): LK[norm(fid[2:])] = ("state", [fid])
# for k, v in REGIONS.items():
#     mem = ["S:" + n for n in v if "S:" + n in cent]
#     if mem: LK[norm(k)] = ("region", mem)

# def mk(kind, mem):
#     return {"kind": kind, "members": mem, "lat": sum(cent[m][0] for m in mem) / len(mem), "lon": sum(cent[m][1] for m in mem) / len(mem)}
# def resolve_one(s):
#     k = norm(s)
#     if not k: return None, ""
#     if k in GLOBAL: return {"kind": "global", "members": ["*"], "lat": 0.0, "lon": -30.0}, "exact"
#     if k in LK: return mk(*LK[k]), "exact"
#     st = s.strip()
#     if st in ab2name and "S:" + ab2name[st] in cent: return mk("state", ["S:" + ab2name[st]]), "abbr"
#     for key in sorted(LK, key=len, reverse=True):
#         if len(key) >= 4 and re.search(r"\b" + re.escape(key) + r"\b", k): return mk(*LK[key]), "contains '" + key + "'"
#     c = difflib.get_close_matches(k, list(LK), n=1, cutoff=GEO_FUZZY)
#     if c: return mk(*LK[c[0]]), "fuzzy '" + c[0] + "'"
#     return None, ""

# unresolved, soft = [], []
# for nid, n in nodes.items():
#     if n["col"] not in GEO_COLS: continue
#     for s in [n["label"]] + n["aliases"]:
#         g, how = resolve_one(s)
#         if g:
#             n["geo"] = g
#             if how != "exact": soft.append("%s -> %s [%s]" % (s, ",".join(m[2:] for m in g["members"][:4]), how))
#             break
#     else: unresolved.append(n["label"])
# if soft: print("geo non-exact matches (verify):\n  " + "\n  ".join(soft))
# if unresolved: print("geo UNRESOLVED (add to REGIONS / ALIASES):", unresolved)

# # ───────── TIME attrs on nodes + rows payload ─────────
# unparsed = []
# for nid, n in nodes.items():
#     if n["col"] in TIME_COLS:
#         for s in [n["label"]] + n["aliases"]:
#             pt = parse_time(s)
#             if pt: n[pt[0]] = pt[1]; break
#         else: unparsed.append(n["label"])
# if unparsed: print("temporal UNPARSED (kept as nodes, ignored by timeline):", unparsed)

# rows = []
# for ids in rows_ids:
#     per, dur = [], []
#     for nid in ids:
#         n = nodes[nid]
#         if "per" in n and n["per"] not in per: per.append(n["per"])
#         if "dur" in n and n["dur"] not in dur: dur.append(n["dur"])
#     geo = sorted({m for nid in ids if "geo" in nodes[nid] for m in nodes[nid]["geo"]["members"]})
#     rows.append({"ids": ids, "geo": geo, "per": per, "dur": dur})
# allp = [p for r in rows for p in r["per"]]; alld = [d for r in rows for d in r["dur"]]
# tmin = min(p[0] for p in allp) if allp else None; tmax = max(p[1] for p in allp) if allp else None
# hmin = max(min(d[0] for d in alld), 1/60) if alld else None; hmax = max(d[1] for d in alld) if alld else None
# if hmin and hmax <= hmin: hmax = hmin * 2
# print("rows w/ geo: %d/%d | rows w/ period: %d | rows w/ cadence: %d" %
#       (sum(bool(r["geo"]) for r in rows), len(rows), sum(bool(r["per"]) for r in rows), sum(bool(r["dur"]) for r in rows)))

# used_c = {m for n in nodes.values() if "geo" in n for m in n["geo"]["members"] if m.startswith("C:")}
# fc = lambda fids: {"type": "FeatureCollection", "features": [FEAT[f] for f in fids if f in FEAT]}
# colors = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(COLS)}
# cnt = Counter(n["col"] for n in nodes.values())
# data = {"cols": COLS, "colors": colors, "counts": {c: cnt[c] for c in COLS}, "hub": HUB_COL,
#         "nodes": list(nodes.values()), "rows": rows,
#         "geo": {"states": fc([f for f in FEAT if f.startswith("S:")]), "countries": fc(sorted(used_c))},
#         "tmin": tmin, "tmax": tmax, "hmin": hmin, "hmax": hmax}

# # ───────── HTML ─────────
# HTML = r"""<!doctype html><html><head><meta charset="utf-8"><title>Network v2</title>
# <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
# <style>
# html,body{margin:0;height:100%;background:#0b0e14;color:#dde3ee;font:12px/1.35 system-ui,sans-serif;overflow:hidden}
# #graph,#map{position:fixed;inset:0}#map{display:none;z-index:0;background:#0b0e14}
# .p{position:fixed;top:8px;background:#141923ee;border:1px solid #2a3142;border-radius:8px;padding:10px;max-height:calc(100vh - 36px);overflow-y:auto;z-index:1000}
# #panel{left:8px;width:262px;max-height:calc(100vh - 36px)}
# #info{right:8px;width:290px;display:none;max-height:calc(100vh - 190px)}
# #tl{left:278px;right:8px;top:auto;bottom:8px;display:grid;grid-template-columns:1fr 1fr;gap:16px;max-height:none;overflow:visible}
# h4{margin:10px 0 4px;font-size:11px;text-transform:uppercase;letter-spacing:.06em;color:#8a93a6}h4:first-child{margin-top:0}
# label.col{display:flex;align-items:center;gap:6px;padding:1px 0;cursor:pointer}
# label.col i{width:10px;height:10px;border-radius:50%;display:inline-block}label.col span{margin-left:auto;color:#6b7488}
# input[type=range]{width:100%;margin:2px 0}select,input[type=text],button{background:#1d2431;color:#dde3ee;border:1px solid #333c50;border-radius:5px;padding:4px 6px;font-size:12px}
# select,input[type=text]{width:100%;box-sizing:border-box}button{cursor:pointer}button:hover{background:#2a3446}button.on{background:#2ec4b6;color:#000;border-color:#2ec4b6}
# .row{display:flex;gap:4px;margin:3px 0;align-items:center}.row>*{flex:1}.v{float:right;color:#8a93a6}
# #info a{color:#7cc4ff;cursor:pointer;display:block;padding:1px 0}#info a:hover{text-decoration:underline}
# #stats{position:fixed;top:8px;left:50%;transform:translateX(-50%);background:#141923cc;padding:4px 10px;border-radius:12px;color:#8a93a6;z-index:1000}
# .chip{display:inline-block;background:#2a3446;border:1px solid #ffd166;border-radius:10px;padding:1px 7px;margin:2px 2px 0 0;cursor:pointer}
# canvas.h{width:100%;height:44px;display:block}.leaflet-container{font:inherit}
# </style></head><body>
# <div id="graph"></div><div id="map"></div>
# <div id="panel" class="p">
#  <div class="row"><button id="vNet" class="on">3D network</button><button id="vMap">Map</button></div>
#  <label class="col"><input type="checkbox" id="pin"> pin geography to map (3D view)</label>
#  <h4>Search</h4><input type="text" id="q" list="dl" placeholder="node / alias, Enter"><datalist id="dl"></datalist>
#  <h4>Geography filter <span class="v"><a href="#" id="gclr">clear</a></span></h4>
#  <select id="gadd"><option value="">+ add place…</option></select><div id="chips"></div>
#  <div style="color:#6b7488">or click states/countries on the Map view</div>
#  <h4>Columns <span class="v"><a href="#" id="all">all</a> · <a href="#" id="none">none</a></span></h4><div id="legend"></div>
#  <h4>Link mode</h4>
#  <select id="mode"><option value="all">All pairs in a row</option><option value="hub">Hub column → everything</option><option value="chain">Chain by column order</option></select>
#  <div class="row"><select id="hub"></select></div>
#  <label class="col"><input type="checkbox" id="intra" checked> link values within same column</label>
#  <h4>Filters</h4>
#  <div>Min edge weight <span class="v" id="vW">1</span></div><input type="range" id="minW" min="1" max="20" value="1">
#  <div>Min node degree <span class="v" id="vD">0</span></div><input type="range" id="minDeg" min="0" max="30" value="0">
#  <h4>Appearance</h4>
#  <label class="col"><input type="checkbox" id="labels"> text labels (3D)</label>
#  <div>Label min degree <span class="v" id="vL">0</span></div><input type="range" id="labelDeg" min="0" max="30" value="0">
#  <div>Node size</div><input type="range" id="size" min="0.3" max="4" step="0.1" value="1">
#  <div>Link opacity (3D)</div><input type="range" id="op" min="0.05" max="1" step="0.05" value="0.35">
#  <label class="col"><input type="checkbox" id="part"> flow particles</label>
#  <label class="col"><input type="checkbox" id="rot"> auto-rotate</label>
#  <h4>Physics (3D)</h4>
#  <div>Link distance</div><input type="range" id="dist" min="10" max="300" value="40">
#  <div>Repulsion</div><input type="range" id="chg" min="5" max="400" value="30">
#  <div class="row"><button id="dim">2D / 3D</button><button id="fit">Fit view</button><button id="snap">PNG</button></div>
# </div>
# <div id="info" class="p"></div><div id="stats"></div>
# <div id="tl" class="p">
#  <div id="pbox"><b>Period (years)</b> <span class="v" id="pLbl"></span>
#   <canvas id="hist" class="h"></canvas><input type="range" id="y0"><input type="range" id="y1">
#   <div class="row"><button id="play">▶ Play</button>
#    <select id="win"><option value="1">1-yr window</option><option value="3" selected>3-yr</option><option value="5">5-yr</option><option value="10">10-yr</option></select>
#    <button id="treset">Reset</button></div>
#   <label class="col"><input type="checkbox" id="incU" checked> include rows without a period</label></div>
#  <div id="cbox"><b>Cadence / lead time</b> <span class="v" id="cLbl"></span>
#   <canvas id="hist2" class="h"></canvas><input type="range" id="h0" min="0" max="100" value="0"><input type="range" id="h1" min="0" max="100" value="100">
#   <label class="col"><input type="checkbox" id="incC" checked> include rows without a cadence</label></div>
# </div>
# <script src="https://unpkg.com/three@0.150.1/build/three.min.js"></script>
# <script src="https://unpkg.com/three-spritetext@1.8.0/dist/three-spritetext.min.js"></script>
# <script src="https://unpkg.com/3d-force-graph@1.73.2/dist/3d-force-graph.min.js"></script>
# <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
# <script>
# const D=__DATA__, $=id=>document.getElementById(id);
# const esc=s=>String(s).replace(/[&<>"]/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));
# const NODE=new Map(D.nodes.map(n=>[n.id,n])), cache=new Map(), CUR={links:[],nodes:[],act:[],anch:new Map(),ver:0,lay:null};
# const S={vis:new Set(D.cols),mode:'all',hub:D.hub,minW:1,minDeg:0,intra:true,labels:false,labelDeg:0,size:1,hl:null,sel:null,dim:3,view:'net',pin:false,
#  geoSel:new Set(),y0:D.tmin,y1:D.tmax,incU:true,h0:D.hmin,h1:D.hmax,incC:true};
# const sid=x=>typeof x==='object'?x.id:x;
# const K=Math.cos(38*Math.PI/180), SC=2.2, proj=(lat,lon)=>[lon*K*SC,lat*SC];
# const hsh=s=>{let h=2166136261;for(let i=0;i<s.length;i++){h^=s.charCodeAt(i);h=Math.imul(h,16777619)}return(h>>>0)/4294967295};
# const jit=(id,k)=>[(hsh(id+'a')-.5)*k,(hsh(id+'b')-.5)*k];
# const fmtH=h=>h<1?Math.round(h*60)+' min':h<48?(+h.toFixed(1))+' h':h<504?(+(h/24).toFixed(1))+' d':h<2160?(+(h/168).toFixed(1))+' wk':h<8760?(+(h/730).toFixed(1))+' mo':(+(h/8760).toFixed(1))+' yr';

# // ───────── row filters (geo + time) ─────────
# const ov=(a0,a1,b0,b1)=>a1>=b0&&a0<=b1;
# function passT(r){
#  if(D.tmin!=null){ if(!r.per.length){ if(!S.incU)return false } else if(!r.per.some(p=>ov(p[0],p[1],S.y0,S.y1)))return false }
#  if(D.hmin!=null){ if(!r.dur.length){ if(!S.incC)return false } else if(!r.dur.some(d=>ov(d[0],d[1],S.h0*.9999,S.h1*1.0001)))return false }
#  return true}
# function passG(r){ if(!S.geoSel.size)return true; if(r.geo.includes('*'))return true; return r.geo.some(g=>S.geoSel.has(g)) }

# // ───────── spatial anchors: geo nodes at their centroid; others at mean of the geographies of their rows ─────────
# function anchors(rows){
#  const A=new Map(), sum=new Map();
#  for(const n of D.nodes) if(n.geo&&n.geo.kind!=='global'){const j=jit(n.id,2.4);A.set(n.id,[n.geo.lat+j[0],n.geo.lon+j[1]])}
#  for(const r of rows){
#   const g=r.ids.filter(i=>NODE.get(i).geo&&A.has(i)); if(!g.length)continue;
#   const m=[g.reduce((s,i)=>s+A.get(i)[0],0)/g.length,g.reduce((s,i)=>s+A.get(i)[1],0)/g.length];
#   for(const id of r.ids){ if(NODE.get(id).geo)continue; const s=sum.get(id)||[0,0,0]; s[0]+=m[0];s[1]+=m[1];s[2]++;sum.set(id,s)}
#  }
#  for(const [id,s] of sum)A.set(id,[s[0]/s[2],s[1]/s[2]]);
#  return A}

# // ───────── 3D graph ─────────
# function labelObj(n){
#  if(!S.labels||n.deg<S.labelDeg||typeof SpriteText==='undefined')return null;
#  const s=new SpriteText(n.label,3.2,D.colors[n.col]);
#  s.position.y=Math.cbrt(Math.max(1,Math.sqrt(n.deg+1))*S.size)*4+4;s.material.depthWrite=false;return s}
# const G=ForceGraph3D({controlType:'trackball'})($('graph')).backgroundColor('#0b0e14').nodeRelSize(4)
#  .nodeVal(n=>Math.max(1,Math.sqrt(n.deg+1))*S.size)
#  .nodeColor(n=>S.hl&&!S.hl.n.has(n.id)?'#262b36':D.colors[n.col])
#  .nodeLabel(nodeTip)
#  .linkColor(l=>S.hl&&!S.hl.l.has(l)?'#1b1f29':'#8a93a6').linkOpacity(.35)
#  .linkWidth(l=>S.hl&&S.hl.l.has(l)?2:Math.min(.3+l.w*.35,4)).linkDirectionalParticleWidth(1.6)
#  .nodeThreeObjectExtend(true).nodeThreeObject(labelObj).onNodeClick(pick).onBackgroundClick(()=>select(null));
# const anchorForce=()=>{let ns=[];const f=a=>{if(!S.pin)return;for(const n of ns){if(n.ax==null||n.geo)continue;
#   n.vx+=(n.ax-n.x)*.3*a;n.vy+=(n.ay-n.y)*.3*a;if(S.dim===3)n.vz+=(0-n.z)*.03*a}};f.initialize=x=>{ns=x};return f};
# G.d3Force('anchor',anchorForce());
# let fitted=false;G.onEngineStop(()=>{if(!fitted){fitted=true;G.zoomToFit(600,60)}});
# function nodeTip(n){return `<div style="background:#000d;padding:4px 8px;border-radius:4px;font:12px system-ui"><b>${esc(n.label)}</b><br>${esc(n.col)} · ${n.deg} links · ${n.rows} rows${n.aliases.length?'<br><i>aka '+esc(n.aliases.join(', '))+'</i>':''}</div>`}
# const refresh=()=>{ if(S.view==='map'){renderMap();return} G.nodeColor(G.nodeColor());G.linkColor(G.linkColor());G.linkWidth(G.linkWidth())};

# let OUT=null;
# function outlines(on){
#  const sc=G.scene(); if(OUT){sc.remove(OUT);OUT.geometry.dispose();OUT=null} if(!on)return;
#  const pts=[], ring=r=>{for(let i=0;i<r.length-1;i++){if(Math.abs(r[i][0]-r[i+1][0])>180)continue;
#    const a=proj(r[i][1],r[i][0]),b=proj(r[i+1][1],r[i+1][0]);pts.push(a[0],a[1],0,b[0],b[1],0)}};
#  for(const fc of [D.geo.states,D.geo.countries])for(const f of fc.features){const g=f.geometry;(g.type==='Polygon'?[g.coordinates]:g.coordinates).forEach(p=>p.forEach(ring))}
#  const geo=new THREE.BufferGeometry();geo.setAttribute('position',new THREE.Float32BufferAttribute(pts,3));
#  OUT=new THREE.LineSegments(geo,new THREE.LineBasicMaterial({color:0x3a4a6b,transparent:true,opacity:.75}));sc.add(OUT)}
# function applyPin(){
#  for(const n of CUR.nodes){const a=CUR.anch.get(n.id);
#   if(S.pin&&a){const p=proj(a[0],a[1]);n.ax=p[0];n.ay=p[1];
#    if(n.geo){n.fx=p[0];n.fy=p[1];n.fz=0}else{n.fx=n.fy=n.fz=null;if(n.x==null){n.x=p[0]+Math.random()*4;n.y=p[1]+Math.random()*4;n.z=Math.random()*10}}}
#   else{n.fx=n.fy=n.fz=null;n.ax=n.ay=null}}}

# // ───────── build ─────────
# function build(){
#  const rows=CUR.act=D.rows.filter(r=>passT(r)&&passG(r));
#  const W=new Map(),cnt=new Map(),add=(a,b)=>{if(a===b)return;const k=a<b?a+'\u0001'+b:b+'\u0001'+a;W.set(k,(W.get(k)||0)+1)};
#  const clique=L=>{for(let i=0;i<L.length;i++)for(let j=i+1;j<L.length;j++)add(L[i],L[j])};
#  for(const row of rows){
#   const ids=row.ids.filter(i=>S.vis.has(NODE.get(i).col)), col=i=>NODE.get(i).col;
#   ids.forEach(i=>cnt.set(i,(cnt.get(i)||0)+1));
#   if(S.mode==='all'){for(let i=0;i<ids.length;i++)for(let j=i+1;j<ids.length;j++){if(!S.intra&&col(ids[i])===col(ids[j]))continue;add(ids[i],ids[j])}}
#   else if(S.mode==='hub'){const h=ids.filter(i=>col(i)===S.hub),r=ids.filter(i=>col(i)!==S.hub);h.forEach(a=>r.forEach(b=>add(a,b)))}
#   else{const by={};ids.forEach(i=>(by[col(i)]??=[]).push(i));const cs=D.cols.filter(c=>by[c]);
#    for(let k=0;k<cs.length-1;k++)by[cs[k]].forEach(a=>by[cs[k+1]].forEach(b=>add(a,b)));if(S.intra)cs.forEach(c=>clique(by[c]))}
#  }
#  const raw=[],deg=new Map();
#  for(const [k,w] of W){if(w<S.minW)continue;const [a,b]=k.split('\u0001');raw.push({source:a,target:b,w});deg.set(a,(deg.get(a)||0)+1);deg.set(b,(deg.get(b)||0)+1)}
#  const nodes=D.nodes.filter(n=>S.vis.has(n.col)&&cnt.has(n.id)&&(deg.get(n.id)||0)>=S.minDeg).map(n=>{
#   let o=cache.get(n.id);if(!o){o={...n};cache.set(n.id,o)}o.deg=deg.get(n.id)||0;o.rows=cnt.get(n.id);return o});
#  const keep=new Set(nodes.map(n=>n.id)), links=raw.filter(l=>keep.has(l.source)&&keep.has(l.target));
#  CUR.links=links;CUR.nodes=nodes;CUR.ver++;CUR.anch=anchors(rows);applyPin();
#  G.graphData({nodes,links});G.nodeThreeObject(labelObj);
#  $('stats').textContent=`${nodes.length} nodes · ${links.length} edges · ${rows.length}/${D.rows.length} rows active`;
#  const s=S.sel&&cache.get(S.sel);if(s&&keep.has(s.id))select(s);else select(null);
# }
# function select(n){
#  S.sel=n?n.id:null;
#  if(!n){S.hl=null;$('info').style.display='none';refresh();return}
#  const ns=new Set([n.id]),ls=new Set(),by={};
#  for(const l of CUR.links){const a=sid(l.source),b=sid(l.target);
#   if(a===n.id||b===n.id){ls.add(l);const o=a===n.id?b:a;ns.add(o);(by[NODE.get(o).col]??=[]).push([o,l.w])}}
#  S.hl={n:ns,l:ls};
#  let h=`<h4>${esc(n.col)}</h4><b style="font-size:14px;color:${D.colors[n.col]}">${esc(n.label)}</b>`;
#  if(n.aliases.length)h+=`<div><i>aka ${esc(n.aliases.join(', '))}</i></div>`;
#  h+=`<div>${n.rows} active rows · ${ns.size-1} connections</div>`;
#  if(n.geo)h+=`<div>geo: ${n.geo.kind==='global'?'global':esc(n.geo.members.length>6?n.geo.members.length+' areas':n.geo.members.map(m=>m.slice(2)).join(', '))}</div>`;
#  if(n.per)h+=`<div>period: ${n.per[0]}–${n.per[1]}</div>`;
#  if(n.dur)h+=`<div>cadence: ${fmtH(n.dur[0])}${n.dur[1]!==n.dur[0]?' – '+fmtH(n.dur[1]):''}</div>`;
#  for(const c of D.cols){const L=by[c];if(!L)continue;L.sort((a,b)=>b[1]-a[1]);
#   h+=`<h4><span style="color:${D.colors[c]}">●</span> ${esc(c)} (${L.length})</h4>`+L.map(([o,w])=>`<a data-id="${esc(o)}">${esc(NODE.get(o).label)} <span style="color:#6b7488">×${w}</span></a>`).join('')}
#  $('info').innerHTML=h;$('info').style.display='block';refresh();
# }
# function pick(n){select(n);if(!n)return;
#  if(S.view==='map'){const p=CUR.lay&&CUR.lay.P.get(n.id);if(p)M.flyTo(p,Math.max(M.getZoom(),5));return}
#  const x=n.x||0,y=n.y||0,z=n.z||0,d=Math.hypot(x,y,z)||1,r=1+120/d;G.cameraPosition({x:x*r,y:y*r,z:z*r},{x,y,z},800)}
# $('info').onclick=e=>{const o=cache.get(e.target.dataset.id);if(o)pick(o)};

# // ───────── Map view (Leaflet) ─────────
# let M=null,GL=null,NL=null;const GF={};
# function initMap(){
#  if(M)return;
#  M=L.map('map',{preferCanvas:true}).setView([39,-96],4);
#  L.tileLayer('https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png',{attribution:'© OpenStreetMap © CARTO',maxZoom:12}).addTo(M);
#  GL=L.layerGroup().addTo(M);
#  for(const fc of [D.geo.states,D.geo.countries])
#   L.geoJSON(fc,{style:()=>({weight:.8,color:'#556',fillOpacity:0}),onEachFeature:(f,l)=>{GF[f.properties.fid]=l;
#    l.on('click',()=>toggleGeo(f.properties.fid));l.bindTooltip('',{sticky:true})}}).addTo(GL);
#  NL=L.layerGroup().addTo(M)}
# function updateChoro(){
#  const c={};let mx=1;
#  for(const r of CUR.act)for(const g of r.geo){if(g==='*')continue;c[g]=(c[g]||0)+1;if(c[g]>mx)mx=c[g]}
#  for(const [fid,l] of Object.entries(GF)){const v=c[fid]||0,sel=S.geoSel.has(fid);
#   l.setStyle({fillColor:'#2ec4b6',fillOpacity:v?.12+.5*v/mx:0,weight:sel?2.5:.8,color:sel?'#ffd166':'#556'});
#   l.setTooltipContent(`<b>${esc(l.feature.properties.name)}</b><br>${v} active rows`+(sel?'<br>(filter on)':'<br>click to filter'))}}
# function layout(){
#  if(CUR.lay&&CUR.lay.ver===CUR.ver)return CUR.lay;
#  const pts=CUR.nodes.filter(n=>CUR.anch.has(n.id)).map(n=>{const a=CUR.anch.get(n.id);return{n,x:a[1]*K,y:a[0],ax:a[1]*K,ay:a[0]}});
#  const md=.9,it=pts.length>1500?8:30;
#  for(let k=0;k<it;k++){
#   for(let i=0;i<pts.length;i++){const p=pts[i];for(let j=i+1;j<pts.length;j++){const q=pts[j];let dx=q.x-p.x,dy=q.y-p.y,d=Math.hypot(dx,dy);
#    if(d<md){if(d<1e-6){const a=hsh(p.n.id+q.n.id)*6.283;dx=Math.cos(a);dy=Math.sin(a);d=1e-3}const f=(md-d)/d*.5;p.x-=dx*f;p.y-=dy*f;q.x+=dx*f;q.y+=dy*f}}}
#   for(const p of pts){p.x+=(p.ax-p.x)*.05;p.y+=(p.ay-p.y)*.05}}
#  const P=new Map(pts.map(p=>[p.n.id,[p.y,p.x/K]]));
#  return CUR.lay={ver:CUR.ver,P,pts}}
# function renderMap(){
#  initMap();NL.clearLayers();updateChoro();
#  const {P,pts}=layout();
#  for(const l of CUR.links){const a=P.get(sid(l.source)),b=P.get(sid(l.target));if(!a||!b)continue;const on=S.hl&&S.hl.l.has(l);
#   if(S.hl&&!on)continue;
#   L.polyline([a,b],{weight:on?2:Math.min(.4+l.w*.3,3),color:on?'#ffd166':'#8a93a6',opacity:on?.9:.3,interactive:false}).addTo(NL)}
#  for(const {n} of pts){const dim=S.hl&&!S.hl.n.has(n.id);
#   L.circleMarker(P.get(n.id),{radius:3+Math.sqrt(n.deg+1)*S.size*1.2,color:'#000',weight:.5,fillColor:dim?'#333':D.colors[n.col],fillOpacity:dim?.4:.92})
#    .bindTooltip(nodeTip(n)).on('click',e=>{L.DomEvent.stopPropagation(e);select(n)}).addTo(NL)}
#  const miss=CUR.nodes.length-pts.length;
#  if(miss)$('stats').textContent=$('stats').textContent.split(' · unanchored')[0]+` · unanchored (no geography): ${miss}`}
# function setView(v){
#  S.view=v;$('graph').style.display=v==='net'?'block':'none';$('map').style.display=v==='map'?'block':'none';
#  $('vNet').classList.toggle('on',v==='net');$('vMap').classList.toggle('on',v==='map');
#  if(v==='map'){G.pauseAnimation();initMap();M.invalidateSize();renderMap();if(!M._fitted&&CUR.lay&&CUR.lay.pts.length){M._fitted=1;fitMap()}}
#  else{G.resumeAnimation();G.width(innerWidth).height(innerHeight)}}
# function fitMap(){const {pts}=layout();if(pts.length)M.fitBounds(L.latLngBounds(pts.map(p=>[p.y,p.x/K])),{padding:[60,60],maxZoom:7})}

# // ───────── geography filter UI ─────────
# const gname=fid=>(D.geo.states.features.concat(D.geo.countries.features).find(f=>f.properties.fid===fid)||{properties:{name:fid}}).properties.name;
# function renderChips(){$('chips').innerHTML=[...S.geoSel].map(f=>`<span class="chip" data-f="${esc(f)}">${esc(gname(f))} ×</span>`).join('')}
# function toggleGeo(f){S.geoSel.has(f)?S.geoSel.delete(f):S.geoSel.add(f);renderChips();build()}
# for(const [lab,fc] of [['',D.geo.states],['',D.geo.countries]])
#  [...fc.features].sort((a,b)=>a.properties.name.localeCompare(b.properties.name)).forEach(f=>{const o=document.createElement('option');
#   o.value=f.properties.fid;o.textContent=f.properties.name+(f.properties.fid.startsWith('C:')?' (country)':'');$('gadd').append(o)});
# $('gadd').onchange=e=>{if(e.target.value&&!S.geoSel.has(e.target.value))toggleGeo(e.target.value);e.target.value=''};
# $('chips').onclick=e=>{if(e.target.dataset.f)toggleGeo(e.target.dataset.f)};
# $('gclr').onclick=e=>{e.preventDefault();S.geoSel.clear();renderChips();build()};

# // ───────── timeline UI ─────────
# function hist(cv,bins,lo,hi){const w=cv.width=cv.clientWidth||300,h=cv.height=44,m=Math.max(1,...bins),bw=w/bins.length,x=cv.getContext('2d');
#  x.clearRect(0,0,w,h);bins.forEach((b,i)=>{x.fillStyle=(i>=lo&&i<=hi)?'#2ec4b6':'#3a4256';const bh=b/m*(h-4);x.fillRect(i*bw+1,h-bh,Math.max(1,bw-2),bh)})}
# let YB=[],HB=[],lg0=0,lg1=1;
# const hv=v=>Math.pow(10,lg0+(lg1-lg0)*v/100);
# if(D.tmin!=null){for(let y=D.tmin;y<=D.tmax;y++)YB.push(D.rows.filter(r=>r.per.some(p=>p[0]<=y&&p[1]>=y)).length);
#  for(const id of['y0','y1']){$(id).min=D.tmin;$(id).max=D.tmax;$(id).step=1}$('y0').value=D.tmin;$('y1').value=D.tmax}else $('pbox').style.display='none';
# if(D.hmin!=null){lg0=Math.log10(D.hmin);lg1=Math.log10(D.hmax);const N=40;
#  for(let i=0;i<N;i++){const a=Math.pow(10,lg0+(lg1-lg0)*i/N),b=Math.pow(10,lg0+(lg1-lg0)*(i+1)/N);HB.push(D.rows.filter(r=>r.dur.some(d=>ov(d[0],d[1],a,b))).length)}}
# else $('cbox').style.display='none';
# if(D.tmin==null&&D.hmin==null)$('tl').style.display='none';
# function syncT(){
#  if(D.tmin!=null){S.y0=+$('y0').value;S.y1=+$('y1').value;$('pLbl').textContent=S.y0+'–'+S.y1;hist($('hist'),YB,S.y0-D.tmin,S.y1-D.tmin)}
#  if(D.hmin!=null){const a=+$('h0').value,b=+$('h1').value;S.h0=hv(a);S.h1=hv(b);$('cLbl').textContent=fmtH(S.h0)+' – '+fmtH(S.h1);
#   const lo=Math.floor(a*.4);hist($('hist2'),HB,lo,Math.max(lo,Math.ceil(b*.4)-1))}}
# $('y0').oninput=()=>{if(+$('y0').value>+$('y1').value)$('y1').value=$('y0').value;syncT();build()};
# $('y1').oninput=()=>{if(+$('y1').value<+$('y0').value)$('y0').value=$('y1').value;syncT();build()};
# $('h0').oninput=()=>{if(+$('h0').value>+$('h1').value)$('h1').value=$('h0').value;syncT();build()};
# $('h1').oninput=()=>{if(+$('h1').value<+$('h0').value)$('h0').value=$('h1').value;syncT();build()};
# $('incU').onchange=e=>{S.incU=e.target.checked;build()};$('incC').onchange=e=>{S.incC=e.target.checked;build()};
# const setWin=(a,b)=>{$('y0').value=a;$('y1').value=b;syncT();build()};
# let pt=null;
# $('play').onclick=()=>{if(pt){clearInterval(pt);pt=null;$('play').textContent='▶ Play';return}
#  $('play').textContent='⏸ Pause';let s=D.tmin;setWin(s,Math.min(D.tmax,s+(+$('win').value)-1));
#  pt=setInterval(()=>{const w=+$('win').value;s++;if(s+w-1>D.tmax)s=D.tmin;setWin(s,s+w-1)},900)};
# $('treset').onclick=()=>{if(pt){clearInterval(pt);pt=null;$('play').textContent='▶ Play'}
#  if(D.tmin!=null){$('y0').value=D.tmin;$('y1').value=D.tmax}$('h0').value=0;$('h1').value=100;syncT();build()};
# addEventListener('resize',syncT);

# // ───────── general UI ─────────
# D.cols.forEach(c=>{const l=document.createElement('label');l.className='col';
#  l.innerHTML=`<input type="checkbox" checked data-c="${esc(c)}"><i style="background:${D.colors[c]}"></i>${esc(c)}<span>${D.counts[c]}</span>`;$('legend').append(l);
#  const o=document.createElement('option');o.value=o.textContent=c;if(c===D.hub)o.selected=true;$('hub').append(o)});
# D.nodes.forEach(n=>{const o=document.createElement('option');o.value=n.label;$('dl').append(o)});
# $('legend').onchange=e=>{const c=e.target.dataset.c;e.target.checked?S.vis.add(c):S.vis.delete(c);build()};
# const setAll=v=>{document.querySelectorAll('#legend input').forEach(i=>{i.checked=v;v?S.vis.add(i.dataset.c):S.vis.delete(i.dataset.c)});build()};
# $('all').onclick=e=>{e.preventDefault();setAll(true)};$('none').onclick=e=>{e.preventDefault();setAll(false)};
# $('mode').onchange=e=>{S.mode=e.target.value;build()};$('hub').onchange=e=>{S.hub=e.target.value;build()};
# $('intra').onchange=e=>{S.intra=e.target.checked;build()};
# $('minW').oninput=e=>{S.minW=+e.target.value;$('vW').textContent=S.minW;build()};
# $('minDeg').oninput=e=>{S.minDeg=+e.target.value;$('vD').textContent=S.minDeg;build()};
# $('labels').onchange=e=>{S.labels=e.target.checked;G.nodeThreeObject(labelObj)};
# $('labelDeg').oninput=e=>{S.labelDeg=+e.target.value;$('vL').textContent=S.labelDeg;G.nodeThreeObject(labelObj)};
# $('size').oninput=e=>{S.size=+e.target.value;G.nodeVal(G.nodeVal());G.nodeThreeObject(labelObj);if(S.view==='map')renderMap()};
# $('op').oninput=e=>G.linkOpacity(+e.target.value);
# $('part').onchange=e=>G.linkDirectionalParticles(e.target.checked?2:0);
# $('dist').oninput=e=>{G.d3Force('link').distance(+e.target.value);G.d3ReheatSimulation()};
# $('chg').oninput=e=>{G.d3Force('charge').strength(-+e.target.value);G.d3ReheatSimulation()};
# let rot=null;$('rot').onchange=e=>{clearInterval(rot);if(e.target.checked)rot=setInterval(()=>{
#  const p=G.cameraPosition(),c=Math.cos(.004),s=Math.sin(.004);G.cameraPosition({x:p.x*c+p.z*s,y:p.y,z:-p.x*s+p.z*c})},30)};
# $('dim').onclick=()=>{S.dim=S.dim===3?2:3;G.numDimensions(S.dim);G.d3ReheatSimulation()};
# $('fit').onclick=()=>S.view==='map'?fitMap():G.zoomToFit(600,60);
# $('snap').onclick=()=>{if(S.view==='map'){alert('PNG export is for the 3D view; use a screenshot for the map.');return}
#  const r=G.renderer();r.render(G.scene(),G.camera());const a=document.createElement('a');a.href=r.domElement.toDataURL('image/png');a.download='network.png';a.click()};
# $('pin').onchange=e=>{S.pin=e.target.checked;outlines(S.pin);applyPin();G.d3ReheatSimulation();setTimeout(()=>G.zoomToFit(800,60),1500)};
# $('vNet').onclick=()=>setView('net');$('vMap').onclick=()=>setView('map');
# $('q').addEventListener('change',e=>{
#  const q=e.target.value.trim().toLowerCase();if(!q)return;
#  const n=D.nodes.find(x=>x.label.toLowerCase()===q)||D.nodes.find(x=>(x.label+' '+x.aliases.join(' ')).toLowerCase().includes(q));
#  if(!n){$('stats').textContent='no match';return}
#  if(!S.vis.has(n.col)){S.vis.add(n.col);document.querySelector(`#legend input[data-c="${CSS.escape(n.col)}"]`).checked=true;build()}
#  setTimeout(()=>{const o=cache.get(n.id);CUR.nodes.includes(o)?pick(o):$('stats').textContent='node hidden by filters'},300)});
# G.d3Force('link').distance(40);G.d3Force('charge').strength(-30);
# syncT();build();
# </script></body></html>"""

# html = HTML.replace("__DATA__", json.dumps(data, ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/"))
# open(OUT_HTML, "w", encoding="utf-8").write(html)
# print("✔ %s: %d nodes, %d rows, %.0f KB" % (OUT_HTML, len(nodes), len(rows), len(html) / 1024))
# try:
#     from google.colab import files; files.download(OUT_HTML)
# except Exception: pass

Saving Network_Viz_A9.csv to Network_Viz_A9.csv
999 rows | sep = COMMA | geo: ['State', 'Area of Interest', 'Area of Interest_2'] | time: ['Temporal', 'Temporal_2']
  fuzzy merge [Area of Interest_2]: 'california' ~ 'califronia' (0.90)
  fuzzy merge [End users]: 'us forest service' ~ 'usda forest service' (0.94)
geo non-exact matches (verify):
  Latvia and Sweden -> Latvia [contains 'latvia']
rows w/ geo: 15/999 | rows w/ period: 8 | rows w/ cadence: 9
✔ network_v2.html: 283 nodes, 999 rows, 179 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# # ── Cell 1 ──
# !pip install pyvis pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 80.5 MB/s eta 0:00:00


In [ ]:
# # ── Cell 2 ──
# import pandas as pd, re, json
# from pyvis.network import Network
# from google.colab import files

# # Upload CSV (or set CSV_PATH to a Drive/GitHub raw path instead)
# uploaded = files.upload()
# CSV_PATH = list(uploaded.keys())[0]

# COLOR_MAP = {
#     "PI": "#e63946", "EA Element": "#f1a208", "Model": "#2a9d8f",
#     "Framework": "#264653", "Inference": "#e76f51", "Output": "#8ecae6",
#     "Temporal": "#ffb703", "Temporal_2": "#fb8500", "Prediction scale": "#6a4c93",
#     "End users": "#4361ee", "End user sector": "#4cc9f0", "Synergies": "#7209b7",
# }
# ATTR_COLS = [c for c in COLOR_MAP if c != "PI"]

# def clean_val(v):
#     """Split multi-entry cells on commas (ignoring commas inside parens), strip cruft."""
#     if pd.isna(v):
#         return []
#     parts = re.split(r",(?![^()]*\))", str(v))
#     out = []
#     for p in parts:
#         p = p.strip(" ,")
#         p = re.sub(r"^\)|\)$", "", p).strip()
#         p = re.sub(r"^and\s+", "", p, flags=re.I).strip()
#         if p:
#             out.append(p)
#     return out

# df = pd.read_csv(CSV_PATH)
# df.columns = [c.strip() for c in df.columns]

# net = Network(height="900px", width="100%", bgcolor="#111111", font_color="white",
#               directed=False, notebook=False, cdn_resources="in_line")
# net.barnes_hut(gravity=-3000, central_gravity=0.3, spring_length=120,
#                spring_strength=0.02, damping=0.09)

# added = {}
# def add_node(label, group):
#     key = f"{group}::{label}"
#     if key not in added:
#         net.add_node(key, label=label, group=group, title=f"{group}: {label}",
#                       color=COLOR_MAP.get(group, "#999999"))
#         added[key] = True
#     return key

# for _, row in df.iterrows():
#     pi_vals = clean_val(row.get("PI"))
#     if not pi_vals:
#         continue
#     pi_key = add_node(pi_vals[0], "PI")
#     for col in ATTR_COLS:
#         for val in clean_val(row.get(col)):
#             node_key = add_node(val, col)
#             net.add_edge(pi_key, node_key, title=col)

# net.set_options(json.dumps({
#     "nodes": {"shape": "dot", "size": 14, "font": {"size": 14, "color": "#ffffff"}},
#     "edges": {"color": {"color": "#555555", "opacity": 0.5}, "smooth": False},
#     "physics": {"stabilization": {"iterations": 200}},
#     "interaction": {"hover": True, "tooltipDelay": 100, "navigationButtons": True, "keyboard": True}
# }))

# OUT_HTML = "network_viz.html"
# net.write_html(OUT_HTML, notebook=False)

# # ── inject filter/search/physics control panel ──
# groups = list(COLOR_MAP.keys())
# checkboxes = "".join(
#     f'<label style="display:block;margin:2px 0;">'
#     f'<input type="checkbox" checked data-group="{g}" '
#     f'onchange="toggleGroup(\'{g}\', this.checked)"> '
#     f'<span style="color:{COLOR_MAP[g]}">●</span> {g}</label>'
#     for g in groups
# )

# panel = f"""
# <div id="controls" style="position:fixed;top:10px;left:10px;z-index:999;
# background:rgba(20,20,20,0.92);color:#fff;padding:14px;border-radius:8px;
# font-family:sans-serif;font-size:13px;max-height:90vh;overflow-y:auto;width:230px;">
#   <b>Filter Groups</b><hr>
#   {checkboxes}
#   <hr>
#   <input id="searchBox" placeholder="Search node..."
#     style="width:100%;margin-bottom:6px;box-sizing:border-box;">
#   <button onclick="doSearch()">Find</button>
#   <button onclick="resetView()">Reset</button><br><br>
#   <label><input type="checkbox" id="physicsToggle" checked onchange="togglePhysics()"> Physics</label>
# </div>
# <script>
# function toggleGroup(g, checked) {{
#   var ids = nodes.getIds().filter(id => nodes.get(id).group === g);
#   nodes.update(ids.map(id => ({{id: id, hidden: !checked}})));
# }}
# function togglePhysics() {{
#   network.setOptions({{physics: {{enabled: document.getElementById('physicsToggle').checked}}}});
# }}
# function doSearch() {{
#   var q = document.getElementById('searchBox').value.toLowerCase();
#   var match = nodes.get().find(n => n.label.toLowerCase().includes(q));
#   if (match) {{
#     network.selectNodes([match.id]);
#     network.focus(match.id, {{scale: 1.5, animation: true}});
#   }}
# }}
# function resetView() {{
#   nodes.update(nodes.getIds().map(id => ({{id: id, hidden: false}})));
#   document.querySelectorAll('#controls input[data-group]').forEach(cb => cb.checked = true);
#   document.getElementById('physicsToggle').checked = true;
#   network.setOptions({{physics: {{enabled: true}}}});
#   network.fit();
# }}
# </script>
# """

# with open(OUT_HTML, "r", encoding="utf-8") as f:
#     html = f.read()
# html = html.replace("</body>", panel + "</body>")
# with open(OUT_HTML, "w", encoding="utf-8") as f:
#     f.write(html)

# files.download(OUT_HTML)
# print(f"Wrote {OUT_HTML}, nodes={len(added)}, edges={len(net.edges)}")

Saving Network_Viz_A9.xlsx to Network_Viz_A9.xlsx


UnicodeDecodeError: 'utf-8' codec can't decode bytes in position 15-16: invalid continuation byte